In [1]:
import requests
import pandas as pd
import numpy as np
import json
import time

print("Environment ready")

Environment ready


In [2]:
import requests
import time

url = "https://api.gdeltproject.org/api/v2/doc/doc"

params = {
    "query": "cyberattack",
    "mode": "artlist",
    "format": "json",
    "maxrecords": 25,
    "timespan": "1week"
}

headers = {
    "User-Agent": "EuropeanSecurityMonitor/1.0"
}

def gdelt_request(url, params, headers, max_retries=3):
    wait_time = 10

    for attempt in range(max_retries):
        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=30
        )

        print(
            f"Attempt {attempt + 1} | "
            f"Status: {response.status_code}"
        )

        if response.status_code == 200:
            print("Request successful")
            return response

        elif response.status_code == 429:
            if attempt < max_retries - 1:
                print(
                    f"Rate limit active. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)
                wait_time *= 2

        else:
            print(response.text[:300])
            response.raise_for_status()

    print("Maximum retries reached.")
    return None

In [3]:
print("GDELT API client ready")

GDELT API client ready


## 2. Security domain taxonomy

In [4]:
SECURITY_DOMAINS = {
    "Defence & Military": [
        "military",
        "defence",
        "defense",
        "army",
        "armed forces",
        "missile",
        "weapon",
        "weapons",
        "air defence",
        "military exercise",
        "troops"
    ],

    "Cybersecurity": [
        "cyberattack",
        "cyber attack",
        "ransomware",
        "malware",
        "cybersecurity",
        "data breach",
        "hacking"
    ],

    "Energy Security": [
        "energy security",
        "gas supply",
        "oil supply",
        "pipeline",
        "electricity grid",
        "energy infrastructure"
    ],

    "Sanctions & Economic Security": [
        "sanctions",
        "export controls",
        "economic sanctions",
        "trade restrictions",
        "asset freeze"
    ],

    "Conflict & Geopolitical Tensions": [
        "conflict",
        "war",
        "invasion",
        "border tensions",
        "military escalation",
        "ceasefire",
        "hostilities"
    ]
}

In [5]:
for domain, keywords in SECURITY_DOMAINS.items():
    print(domain)
    print("  ", keywords)

Defence & Military
   ['military', 'defence', 'defense', 'army', 'armed forces', 'missile', 'weapon', 'weapons', 'air defence', 'military exercise', 'troops']
Cybersecurity
   ['cyberattack', 'cyber attack', 'ransomware', 'malware', 'cybersecurity', 'data breach', 'hacking']
Energy Security
   ['energy security', 'gas supply', 'oil supply', 'pipeline', 'electricity grid', 'energy infrastructure']
Sanctions & Economic Security
   ['sanctions', 'export controls', 'economic sanctions', 'trade restrictions', 'asset freeze']
Conflict & Geopolitical Tensions
   ['conflict', 'war', 'invasion', 'border tensions', 'military escalation', 'ceasefire', 'hostilities']


## 3. Rule-Based Security Classification

In [6]:
def classify_security_domain(text, domains):
    """
    Classifies a text into one or more security domains
    based on keyword matches.
    """
    
    if not isinstance(text, str):
        return ["Unclassified"]

    text = text.lower()

    matches = []

    for domain, keywords in domains.items():
        for keyword in keywords:
            if keyword.lower() in text:
                matches.append(domain)
                break

    if not matches:
        return ["Unclassified"]

    return matches

In [7]:
test_text = "Estonia reports major cyberattack against government systems"

classify_security_domain(
    test_text,
    SECURITY_DOMAINS
)

['Cybersecurity']

In [8]:
test_articles = [
    "Estonia reports major cyberattack against government systems",
    "Poland announces new air defence procurement programme",
    "European Union approves new sanctions against Russia",
    "Damage to gas pipeline raises energy security concerns",
    "Military escalation continues near the Ukrainian border",
    "European leaders meet in Brussels to discuss migration"
]

for article in test_articles:
    classification = classify_security_domain(
        article,
        SECURITY_DOMAINS
    )
    
    print(article)
    print("Classification:", classification)
    print("-" * 80)

Estonia reports major cyberattack against government systems
Classification: ['Cybersecurity']
--------------------------------------------------------------------------------
Poland announces new air defence procurement programme
Classification: ['Defence & Military']
--------------------------------------------------------------------------------
European Union approves new sanctions against Russia
Classification: ['Sanctions & Economic Security']
--------------------------------------------------------------------------------
Damage to gas pipeline raises energy security concerns
Classification: ['Energy Security']
--------------------------------------------------------------------------------
Military escalation continues near the Ukrainian border
Classification: ['Defence & Military', 'Conflict & Geopolitical Tensions']
--------------------------------------------------------------------------------
European leaders meet in Brussels to discuss migration
Classification: ['Unclassi

In [9]:
df_test = pd.DataFrame({
    "title": test_articles
})

df_test

,title
0,Estonia reports major cyberattack against gove...
1,Poland announces new air defence procurement p...
2,European Union approves new sanctions against ...
3,Damage to gas pipeline raises energy security ...
4,Military escalation continues near the Ukraini...
5,European leaders meet in Brussels to discuss m...


In [10]:
df_test["security_domain"] = df_test["title"].apply(
    lambda x: classify_security_domain(
        x,
        SECURITY_DOMAINS
    )
)

df_test

,title,security_domain
0,Estonia reports major cyberattack against gove...,[Cybersecurity]
1,Poland announces new air defence procurement p...,[Defence & Military]
2,European Union approves new sanctions against ...,[Sanctions & Economic Security]
3,Damage to gas pipeline raises energy security ...,[Energy Security]
4,Military escalation continues near the Ukraini...,"[Defence & Military, Conflict & Geopolitical T..."
5,European leaders meet in Brussels to discuss m...,[Unclassified]


In [11]:
def find_security_keywords(text, domains):
    """
    Returns the security keywords detected in a text.
    """
    
    if not isinstance(text, str):
        return []

    text = text.lower()

    detected_keywords = []

    for domain, keywords in domains.items():
        for keyword in keywords:
            if keyword.lower() in text:
                detected_keywords.append(keyword)

    return list(set(detected_keywords))

In [12]:
df_test["matched_keywords"] = df_test["title"].apply(
    lambda x: find_security_keywords(
        x,
        SECURITY_DOMAINS
    )
)

df_test

,title,security_domain,matched_keywords
0,Estonia reports major cyberattack against gove...,[Cybersecurity],[cyberattack]
1,Poland announces new air defence procurement p...,[Defence & Military],"[defence, air defence]"
2,European Union approves new sanctions against ...,[Sanctions & Economic Security],[sanctions]
3,Damage to gas pipeline raises energy security ...,[Energy Security],"[energy security, pipeline]"
4,Military escalation continues near the Ukraini...,"[Defence & Military, Conflict & Geopolitical T...","[military, military escalation]"
5,European leaders meet in Brussels to discuss m...,[Unclassified],[]


## 4. Country Entity Extraction with spaCy

In [13]:
import sys

print(sys.executable)

c:\Users\cl_am\AppData\Local\Programs\Python\Python314\python.exe


In [14]:
EUROPEAN_COUNTRIES = [
    "Albania",
    "Austria",
    "Andorra",
    "Belarus",
    "Belgium",
    "Bosnia and Herzegovina",
    "Bulgaria",
    "Croatia",
    "Cyprus",
    "Czechia",
    "Denmark",
    "Estonia",
    "Finland",
    "France",
    "Germany",
    "Greece",
    "Hungary",
    "Iceland",
    "Ireland",
    "Italy",
    "Kosovo",
    "Latvia",
    "Liechtenstein",
    "Lithuania",
    "Luxembourg",
    "Malta",
    "Moldova",
    "Monaco",
    "Montenegro",
    "Netherlands",
    "North Macedonia",
    "Norway",
    "Poland",
    "Portugal",
    "Romania",
    "San Marino",
    "Serbia",
    "Slovakia",
    "Slovenia",
    "Spain",
    "Sweden",
    "Switzerland",
    "Türkiye",
    "Ukraine",
    "United Kingdom"
    "Vatican City",
]

STRATEGIC_NEIGHBOURS = [
    "Russia",
    "Georgia",
    "Armenia",
    "Azerbaijan"
]

MONITORED_COUNTRIES = EUROPEAN_COUNTRIES + STRATEGIC_NEIGHBOURS

print(len(MONITORED_COUNTRIES))

49


In [15]:

COUNTRY_ALIASES = {
    "UK": "United Kingdom",
    "U.K.": "United Kingdom",
    "Britain": "United Kingdom",
    "British": "United Kingdom",

    "Turkey": "Türkiye",
    "Turkish": "Türkiye",

    "Czech Republic": "Czechia",

    "Macedonia": "North Macedonia",

    "Russian": "Russia",
    "Russians": "Russia",

    "Ukrainian": "Ukraine",
    "Ukrainians": "Ukraine",

    "Belarusian": "Belarus",

    "Polish": "Poland",
    "German": "Germany",
    "French": "France",
    "Spanish": "Spain",
    "Italian": "Italy",
    "Estonian": "Estonia",
    "Latvian": "Latvia",
    "Lithuanian": "Lithuania",
    "Finnish": "Finland",
    "Swedish": "Sweden",
    "Norwegian": "Norway",
    "Danish": "Denmark"
}

In [16]:
import re

def extract_countries(text):
    """
    Detects monitored countries mentioned in text
    using country names and aliases.
    """

    if not isinstance(text, str):
        return []

    detected = []

    # Direct country names
    for country in MONITORED_COUNTRIES:

        pattern = rf"\b{re.escape(country)}\b"

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            detected.append(country)

    # Country aliases
    for alias, country in COUNTRY_ALIASES.items():

        pattern = rf"\b{re.escape(alias)}\b"

        if re.search(
            pattern,
            text,
            flags=re.IGNORECASE
        ):
            detected.append(country)

    # Remove duplicates while preserving order
    return list(dict.fromkeys(detected))

In [17]:
country_tests = [
    "Poland announces new air defence procurement programme",
    "Estonia reports major cyberattack against government systems",
    "Germany and France discuss new defence cooperation",
    "British forces deploy to Poland",
    "Russia launches new attacks against Ukraine",
    "NATO officials meet in Brussels",
    "European leaders discuss defence spending"
]

for article in country_tests:

    countries = extract_countries(article)

    print(article)
    print("Countries:", countries)
    print("-" * 80)

Poland announces new air defence procurement programme
Countries: ['Poland']
--------------------------------------------------------------------------------
Estonia reports major cyberattack against government systems
Countries: ['Estonia']
--------------------------------------------------------------------------------
Germany and France discuss new defence cooperation
Countries: ['France', 'Germany']
--------------------------------------------------------------------------------
British forces deploy to Poland
Countries: ['Poland', 'United Kingdom']
--------------------------------------------------------------------------------
Russia launches new attacks against Ukraine
Countries: ['Ukraine', 'Russia']
--------------------------------------------------------------------------------
NATO officials meet in Brussels
Countries: []
--------------------------------------------------------------------------------
European leaders discuss defence spending
Countries: []
------------------

In [18]:
df_test["countries"] = df_test["title"].apply(
    extract_countries
)

df_test

,title,security_domain,matched_keywords,countries
0,Estonia reports major cyberattack against gove...,[Cybersecurity],[cyberattack],[Estonia]
1,Poland announces new air defence procurement p...,[Defence & Military],"[defence, air defence]",[Poland]
2,European Union approves new sanctions against ...,[Sanctions & Economic Security],[sanctions],[Russia]
3,Damage to gas pipeline raises energy security ...,[Energy Security],"[energy security, pipeline]",[]
4,Military escalation continues near the Ukraini...,"[Defence & Military, Conflict & Geopolitical T...","[military, military escalation]",[Ukraine]
5,European leaders meet in Brussels to discuss m...,[Unclassified],[],[]


## 5. GDELT API Data Acquisition

In [19]:
import requests

url = "https://api.gdeltproject.org/api/v2/doc/doc"

params = {
    "query": "cyberattack",
    "mode": "artlist",
    "format": "json",
    "maxrecords": 25,
    "timespan": "1week"
}

headers = {
    "User-Agent": "EuropeanSecurityMonitor/1.0"
}

response = requests.get(
    url,
    params=params,
    headers=headers,
    timeout=30
)

print("Status code:", response.status_code)

Status code: 200


In [20]:
import requests

last_update_url = "https://data.gdeltproject.org/gdeltv2/lastupdate.txt"

response = requests.get(
    last_update_url,
    timeout=30
)

print("Status code:", response.status_code)
print(response.text)

Status code: 200
76286 3cc7a742d7ded1454716e28ddefa7764 http://data.gdeltproject.org/gdeltv2/20260824131500.export.CSV.zip
95554 1e5089de43638116a184935bcc3a751e http://data.gdeltproject.org/gdeltv2/20260824131500.mentions.CSV.zip
4783018 0087ee30ad55d538ddc9ad1c55ab53f5 http://data.gdeltproject.org/gdeltv2/20260824131500.gkg.csv.zip



In [21]:
lines = response.text.strip().splitlines()

export_line = lines[0]

export_url = export_line.split()[-1]

print("Latest GDELT Events file:")
print(export_url)

Latest GDELT Events file:
http://data.gdeltproject.org/gdeltv2/20260824131500.export.CSV.zip


In [22]:
from pathlib import Path

# Detect project root
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

raw_dir = project_root / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Raw data folder:", raw_dir)

Project root: c:\Users\cl_am\OneDrive\Desktop\european-security-monitor
Raw data folder: c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\raw


In [23]:
# Use HTTPS
export_url = export_url.replace("http://", "https://")

file_name = export_url.split("/")[-1]
zip_path = raw_dir / file_name

gdelt_response = requests.get(
    export_url,
    timeout=60
)

gdelt_response.raise_for_status()

with open(zip_path, "wb") as file:
    file.write(gdelt_response.content)

print("Downloaded:")
print(zip_path)
print("File size:", round(zip_path.stat().st_size / 1024, 2), "KB")

Downloaded:
c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\raw\20260824131500.export.CSV.zip
File size: 74.5 KB


## 6. Load GDELT Events

In [24]:
GDELT_EVENT_COLUMNS = [
    "GLOBALEVENTID",
    "SQLDATE",
    "MonthYear",
    "Year",
    "FractionDate",
    "Actor1Code",
    "Actor1Name",
    "Actor1CountryCode",
    "Actor1KnownGroupCode",
    "Actor1EthnicCode",
    "Actor1Religion1Code",
    "Actor1Religion2Code",
    "Actor1Type1Code",
    "Actor1Type2Code",
    "Actor1Type3Code",
    "Actor2Code",
    "Actor2Name",
    "Actor2CountryCode",
    "Actor2KnownGroupCode",
    "Actor2EthnicCode",
    "Actor2Religion1Code",
    "Actor2Religion2Code",
    "Actor2Type1Code",
    "Actor2Type2Code",
    "Actor2Type3Code",
    "IsRootEvent",
    "EventCode",
    "EventBaseCode",
    "EventRootCode",
    "QuadClass",
    "GoldsteinScale",
    "NumMentions",
    "NumSources",
    "NumArticles",
    "AvgTone",
    "Actor1Geo_Type",
    "Actor1Geo_FullName",
    "Actor1Geo_CountryCode",
    "Actor1Geo_ADM1Code",
    "Actor1Geo_ADM2Code",
    "Actor1Geo_Lat",
    "Actor1Geo_Long",
    "Actor1Geo_FeatureID",
    "Actor2Geo_Type",
    "Actor2Geo_FullName",
    "Actor2Geo_CountryCode",
    "Actor2Geo_ADM1Code",
    "Actor2Geo_ADM2Code",
    "Actor2Geo_Lat",
    "Actor2Geo_Long",
    "Actor2Geo_FeatureID",
    "ActionGeo_Type",
    "ActionGeo_FullName",
    "ActionGeo_CountryCode",
    "ActionGeo_ADM1Code",
    "ActionGeo_ADM2Code",
    "ActionGeo_Lat",
    "ActionGeo_Long",
    "ActionGeo_FeatureID",
    "DATEADDED",
    "SOURCEURL"
]

In [25]:
df_gdelt = pd.read_csv(
    zip_path,
    sep="\t",
    header=None,
    names=GDELT_EVENT_COLUMNS,
    compression="zip",
    low_memory=False
)

print("Rows:", len(df_gdelt))
print("Columns:", len(df_gdelt.columns))

df_gdelt.head()

Rows: 1175
Columns: 61


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1319682889,20250824,202508,2025,2025.6411,NaN,NaN,NaN,NaN,NaN,...,4,"Porto, Porto, Portugal",PO,PO17,24866,41.1496,-8.61099,-2173088,20260824131500,https://www.theportugalnews.com/news/2026-08-2...
1,1319682890,20250824,202508,2025,2025.6411,NaN,NaN,NaN,NaN,NaN,...,4,"Oeiras, Lisboa, Portugal",PO,PO14,24835,38.6910,-9.31085,-2170860,20260824131500,https://www.theportugalnews.com/news/2026-08-2...
2,1319682891,20250824,202508,2025,2025.6411,NaN,NaN,NaN,NaN,NaN,...,4,"Porto, Porto, Portugal",PO,PO17,24866,41.1496,-8.61099,-2173088,20260824131500,https://www.theportugalnews.com/news/2026-08-2...
3,1319682892,20250824,202508,2025,2025.6411,EDU,SCHOOL,NaN,NaN,NaN,...,1,United Kingdom,UK,UK,NaN,54.0000,-4.00000,UK,20260824131500,https://www.dailyrecord.co.uk/lifestyle/money/...
4,1319682893,20250824,202508,2025,2025.6411,GBR,SCOTLAND,GBR,NaN,NaN,...,1,United Kingdom,UK,UK,NaN,54.0000,-4.00000,UK,20260824131500,https://www.dailyrecord.co.uk/lifestyle/money/...


## 7. Initial Event Dataset

In [26]:
selected_columns = [
    "GLOBALEVENTID",
    "SQLDATE",
    "Actor1Name",
    "Actor1CountryCode",
    "Actor2Name",
    "Actor2CountryCode",
    "EventCode",
    "EventRootCode",
    "QuadClass",
    "GoldsteinScale",
    "NumMentions",
    "NumSources",
    "NumArticles",
    "AvgTone",
    "ActionGeo_FullName",
    "ActionGeo_CountryCode",
    "ActionGeo_Lat",
    "ActionGeo_Long",
    "SOURCEURL"
]

df_events = df_gdelt[selected_columns].copy()

In [27]:
df_events = df_events.rename(columns={
    "GLOBALEVENTID": "event_id",
    "SQLDATE": "event_date",
    "Actor1Name": "actor1",
    "Actor1CountryCode": "actor1_country",
    "Actor2Name": "actor2",
    "Actor2CountryCode": "actor2_country",
    "EventCode": "event_code",
    "EventRootCode": "event_root_code",
    "QuadClass": "quad_class",
    "GoldsteinScale": "goldstein_scale",
    "NumMentions": "num_mentions",
    "NumSources": "num_sources",
    "NumArticles": "num_articles",
    "AvgTone": "avg_tone",
    "ActionGeo_FullName": "location",
    "ActionGeo_CountryCode": "location_country",
    "ActionGeo_Lat": "latitude",
    "ActionGeo_Long": "longitude",
    "SOURCEURL": "source_url"
})

In [28]:
df_events["event_date"] = pd.to_datetime(
    df_events["event_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

In [29]:
print("Rows:", len(df_events))
print("Columns:", len(df_events.columns))
print("Date range:")
print(df_events["event_date"].min(), "→", df_events["event_date"].max())

df_events.head(10)

Rows: 1175
Columns: 19
Date range:
2025-08-24 00:00:00 → 2026-08-24 00:00:00


,event_id,event_date,actor1,actor1_country,actor2,actor2_country,event_code,event_root_code,quad_class,goldstein_scale,num_mentions,num_sources,num_articles,avg_tone,location,location_country,latitude,longitude,source_url
0,1319682889,2025-08-24,NaN,NaN,LISBON,PRT,42,4,1,1.9,3,1,3,1.333333,"Porto, Porto, Portugal",PO,41.1496,-8.61099,https://www.theportugalnews.com/news/2026-08-2...
1,1319682890,2025-08-24,NaN,NaN,PORTUGAL,PRT,42,4,1,1.9,4,1,4,1.333333,"Oeiras, Lisboa, Portugal",PO,38.6910,-9.31085,https://www.theportugalnews.com/news/2026-08-2...
2,1319682891,2025-08-24,NaN,NaN,PORTUGAL,PRT,42,4,1,1.9,3,1,3,1.333333,"Porto, Porto, Portugal",PO,41.1496,-8.61099,https://www.theportugalnews.com/news/2026-08-2...
3,1319682892,2025-08-24,SCHOOL,NaN,SCOTLAND,GBR,36,3,1,4.0,10,1,10,4.892966,United Kingdom,UK,54.0000,-4.00000,https://www.dailyrecord.co.uk/lifestyle/money/...
4,1319682893,2025-08-24,SCOTLAND,GBR,SCHOOL,NaN,36,3,1,4.0,10,1,10,4.892966,United Kingdom,UK,54.0000,-4.00000,https://www.dailyrecord.co.uk/lifestyle/money/...
5,1319682894,2025-08-24,SPECIAL COURT,NaN,STUDENT,NaN,173,17,4,-5.0,5,1,5,-0.709750,"State Of Kerala, Kerala, India",IN,10.0000,76.50000,https://www.livelaw.in/high-court/kerala-high-...
6,1319682895,2025-08-24,SPECIAL COURT,NaN,STUDENT,NaN,1821,18,4,-9.0,5,1,5,-0.709750,"State Of Kerala, Kerala, India",IN,10.0000,76.50000,https://www.livelaw.in/high-court/kerala-high-...
7,1319682896,2025-08-24,LISBON,PRT,NaN,NaN,43,4,1,2.8,3,1,3,1.333333,"Porto, Porto, Portugal",PO,41.1496,-8.61099,https://www.theportugalnews.com/news/2026-08-2...
8,1319682897,2025-08-24,PORTUGAL,PRT,NaN,NaN,43,4,1,2.8,4,1,4,1.333333,"Oeiras, Lisboa, Portugal",PO,38.6910,-9.31085,https://www.theportugalnews.com/news/2026-08-2...
9,1319682898,2025-08-24,PORTUGAL,PRT,NaN,NaN,43,4,1,2.8,3,1,3,1.333333,"Porto, Porto, Portugal",PO,41.1496,-8.61099,https://www.theportugalnews.com/news/2026-08-2...


In [30]:
print("df_gdelt rows:", len(df_gdelt))
print("df_events rows:", len(df_events))

print("\ndf_gdelt shape:")
print(df_gdelt.shape)

print("\ndf_events shape:")
print(df_events.shape)

df_gdelt rows: 1175
df_events rows: 1175

df_gdelt shape:
(1175, 61)

df_events shape:
(1175, 19)


In [31]:
df_events.head()

,event_id,event_date,actor1,actor1_country,actor2,actor2_country,event_code,event_root_code,quad_class,goldstein_scale,num_mentions,num_sources,num_articles,avg_tone,location,location_country,latitude,longitude,source_url
0,1319682889,2025-08-24,NaN,NaN,LISBON,PRT,42,4,1,1.9,3,1,3,1.333333,"Porto, Porto, Portugal",PO,41.1496,-8.61099,https://www.theportugalnews.com/news/2026-08-2...
1,1319682890,2025-08-24,NaN,NaN,PORTUGAL,PRT,42,4,1,1.9,4,1,4,1.333333,"Oeiras, Lisboa, Portugal",PO,38.6910,-9.31085,https://www.theportugalnews.com/news/2026-08-2...
2,1319682891,2025-08-24,NaN,NaN,PORTUGAL,PRT,42,4,1,1.9,3,1,3,1.333333,"Porto, Porto, Portugal",PO,41.1496,-8.61099,https://www.theportugalnews.com/news/2026-08-2...
3,1319682892,2025-08-24,SCHOOL,NaN,SCOTLAND,GBR,36,3,1,4.0,10,1,10,4.892966,United Kingdom,UK,54.0000,-4.00000,https://www.dailyrecord.co.uk/lifestyle/money/...
4,1319682893,2025-08-24,SCOTLAND,GBR,SCHOOL,NaN,36,3,1,4.0,10,1,10,4.892966,United Kingdom,UK,54.0000,-4.00000,https://www.dailyrecord.co.uk/lifestyle/money/...


## 8. Geographic Coverage Inspection

In [32]:
print("Actor 1 country codes:")
print(
    sorted(
        df_events["actor1_country"]
        .dropna()
        .astype(str)
        .unique()
    )
)

print("\nActor 2 country codes:")
print(
    sorted(
        df_events["actor2_country"]
        .dropna()
        .astype(str)
        .unique()
    )
)

print("\nLocation country codes:")
print(
    sorted(
        df_events["location_country"]
        .dropna()
        .astype(str)
        .unique()
    )
)

Actor 1 country codes:
['AFR', 'ARG', 'AUS', 'AUT', 'BGR', 'BHR', 'BLR', 'BRA', 'BRN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHN', 'CIV', 'CMR', 'COD', 'CZE', 'DJI', 'DNK', 'EGY', 'EST', 'ETH', 'EUR', 'FIN', 'FRA', 'GBR', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISR', 'ITA', 'JOR', 'JPN', 'KAZ', 'KOR', 'LBY', 'LKA', 'LSO', 'LVA', 'MDA', 'MEX', 'MYS', 'NAM', 'NGA', 'NLD', 'NOR', 'OMN', 'PAK', 'PAN', 'POL', 'PRT', 'PSE', 'QAT', 'RUS', 'SAU', 'SDN', 'SGP', 'SOM', 'SYR', 'TCD', 'THA', 'TUR', 'TZA', 'UKR', 'USA', 'UZB', 'VNM', 'WST', 'YEM', 'ZAF', 'ZMB']

Actor 2 country codes:
['AFR', 'ARE', 'ARG', 'AUS', 'AUT', 'BRN', 'BWA', 'CAF', 'CAN', 'CHE', 'CHN', 'CIV', 'CMR', 'COG', 'CYP', 'CZE', 'DJI', 'DNK', 'DZA', 'EGY', 'EUR', 'FIN', 'FRA', 'GBR', 'GRC', 'IDN', 'IND', 'IRL', 'IRN', 'IRQ', 'ISR', 'ITA', 'JOR', 'JPN', 'KAZ', 'LBN', 'LBY', 'LVA', 'MDA', 'MEX', 'MOZ', 'MYS', 'NAM', 'NGA', 'NLD', 'NOR', 'OMN', 'PAK', 'POL', 'PRT', 'PSE', 'QAT', 'RUS', 'SAU', 'SDN', 'SGP', 'SOM', 'SYR', 'TCD', 'THA', 'TZA', 'UKR',

In [33]:
df_events[
    [
        "actor1",
        "actor1_country",
        "actor2",
        "actor2_country",
        "location",
        "location_country"
    ]
].head(20)

,actor1,actor1_country,actor2,actor2_country,location,location_country
0,NaN,NaN,LISBON,PRT,"Porto, Porto, Portugal",PO
1,NaN,NaN,PORTUGAL,PRT,"Oeiras, Lisboa, Portugal",PO
2,NaN,NaN,PORTUGAL,PRT,"Porto, Porto, Portugal",PO
3,SCHOOL,NaN,SCOTLAND,GBR,United Kingdom,UK
4,SCOTLAND,GBR,SCHOOL,NaN,United Kingdom,UK
5,SPECIAL COURT,NaN,STUDENT,NaN,"State Of Kerala, Kerala, India",IN
6,SPECIAL COURT,NaN,STUDENT,NaN,"State Of Kerala, Kerala, India",IN
7,LISBON,PRT,NaN,NaN,"Porto, Porto, Portugal",PO
8,PORTUGAL,PRT,NaN,NaN,"Oeiras, Lisboa, Portugal",PO
9,PORTUGAL,PRT,NaN,NaN,"Porto, Porto, Portugal",PO


## 9. European & Strategic Event Filtering

In [34]:
MONITORED_ISO3 = {
    "ALB",  # Albania
    "AUT",  # Austria
    "BLR",  # Belarus
    "BEL",  # Belgium
    "BIH",  # Bosnia and Herzegovina
    "BGR",  # Bulgaria
    "HRV",  # Croatia
    "CYP",  # Cyprus
    "CZE",  # Czechia
    "DNK",  # Denmark
    "EST",  # Estonia
    "FIN",  # Finland
    "FRA",  # France
    "DEU",  # Germany
    "GRC",  # Greece
    "HUN",  # Hungary
    "ISL",  # Iceland
    "IRL",  # Ireland
    "ITA",  # Italy
    "LVA",  # Latvia
    "LTU",  # Lithuania
    "LUX",  # Luxembourg
    "MLT",  # Malta
    "MDA",  # Moldova
    "MNE",  # Montenegro
    "NLD",  # Netherlands
    "MKD",  # North Macedonia
    "NOR",  # Norway
    "POL",  # Poland
    "PRT",  # Portugal
    "ROU",  # Romania
    "SRB",  # Serbia
    "SVK",  # Slovakia
    "SVN",  # Slovenia
    "ESP",  # Spain
    "SWE",  # Sweden
    "CHE",  # Switzerland
    "TUR",  # Türkiye
    "UKR",  # Ukraine
    "GBR",  # United Kingdom

    # Strategic neighbourhood
    "RUS",  # Russia
    "GEO",  # Georgia
    "ARM",  # Armenia
    "AZE"   # Azerbaijan
}

In [35]:
df_events["location_countries"] = (
    df_events["location"]
    .apply(extract_countries)
)

In [36]:
df_events[
    [
        "location",
        "location_country",
        "location_countries"
    ]
].head(20)

,location,location_country,location_countries
0,"Porto, Porto, Portugal",PO,[Portugal]
1,"Oeiras, Lisboa, Portugal",PO,[Portugal]
2,"Porto, Porto, Portugal",PO,[Portugal]
3,United Kingdom,UK,[]
4,United Kingdom,UK,[]
5,"State Of Kerala, Kerala, India",IN,[]
6,"State Of Kerala, Kerala, India",IN,[]
7,"Porto, Porto, Portugal",PO,[Portugal]
8,"Oeiras, Lisboa, Portugal",PO,[Portugal]
9,"Porto, Porto, Portugal",PO,[Portugal]


In [37]:
actor1_match = (
    df_events["actor1_country"]
    .isin(MONITORED_ISO3)
)

actor2_match = (
    df_events["actor2_country"]
    .isin(MONITORED_ISO3)
)

location_match = (
    df_events["location_countries"]
    .apply(lambda x: len(x) > 0)
)

df_europe = df_events[
    actor1_match |
    actor2_match |
    location_match
].copy()

print("Global events:", len(df_events))
print("European / strategic events:", len(df_europe))

print(
    "Share retained:",
    round(
        len(df_europe) / len(df_events) * 100,
        2
    ),
    "%"
)

Global events: 1175
European / strategic events: 267
Share retained: 22.72 %


In [38]:
df_europe[
    [
        "event_date",
        "actor1",
        "actor1_country",
        "actor2",
        "actor2_country",
        "location",
        "location_countries"
    ]
].head(20)

,event_date,actor1,actor1_country,actor2,actor2_country,location,location_countries
0,2025-08-24,NaN,NaN,LISBON,PRT,"Porto, Porto, Portugal",[Portugal]
1,2025-08-24,NaN,NaN,PORTUGAL,PRT,"Oeiras, Lisboa, Portugal",[Portugal]
2,2025-08-24,NaN,NaN,PORTUGAL,PRT,"Porto, Porto, Portugal",[Portugal]
3,2025-08-24,SCHOOL,NaN,SCOTLAND,GBR,United Kingdom,[]
4,2025-08-24,SCOTLAND,GBR,SCHOOL,NaN,United Kingdom,[]
7,2025-08-24,LISBON,PRT,NaN,NaN,"Porto, Porto, Portugal",[Portugal]
8,2025-08-24,PORTUGAL,PRT,NaN,NaN,"Oeiras, Lisboa, Portugal",[Portugal]
9,2025-08-24,PORTUGAL,PRT,NaN,NaN,"Porto, Porto, Portugal",[Portugal]
16,2026-08-17,GOVERNMENT SPOKESMAN,NaN,FRANCE,FRA,"Paris, France (general), France",[France]
17,2026-08-17,GOVERNMENT SPOKESMAN,NaN,FRENCH,FRA,"Paris, France (general), France",[France]


In [39]:
LOCATION_COUNTRY_ALIASES = {
    "Turkey": "Türkiye",
    "Czech Republic": "Czechia",
    "Russian Federation": "Russia",
    "Vatican": "Vatican City"
}


def extract_gdelt_location_country(location):
    """
    Extracts the country from a GDELT geographic location.
    GDELT locations usually end with the country name.
    """

    if not isinstance(location, str):
        return []

    # Last component of the GDELT location
    country_candidate = location.split(",")[-1].strip()

    # Normalise aliases
    country_candidate = LOCATION_COUNTRY_ALIASES.get(
        country_candidate,
        country_candidate
    )

    if country_candidate in MONITORED_COUNTRIES:
        return [country_candidate]

    return []

In [40]:
df_events["location_countries"] = (
    df_events["location"]
    .apply(extract_gdelt_location_country)
)

In [41]:
test_locations = [
    "Paris, France (general), France",
    "Port-Of-Spain, Port-of-Spain, Trinidad And Tobago",
    "Scottish Highlands, Highland, United Kingdom",
    "Monaco",
    "Nagoya, Aichi, Japan",
    "Kyiv, Ukraine"
]

for location in test_locations:
    print(
        location,
        "→",
        extract_gdelt_location_country(location)
    )

Paris, France (general), France → ['France']
Port-Of-Spain, Port-of-Spain, Trinidad And Tobago → []
Scottish Highlands, Highland, United Kingdom → []
Monaco → ['Monaco']
Nagoya, Aichi, Japan → []
Kyiv, Ukraine → ['Ukraine']


In [42]:
actor1_match = (
    df_events["actor1_country"]
    .isin(MONITORED_ISO3)
)

actor2_match = (
    df_events["actor2_country"]
    .isin(MONITORED_ISO3)
)

location_match = (
    df_events["location_countries"]
    .apply(lambda x: len(x) > 0)
)

df_europe = df_events[
    actor1_match |
    actor2_match |
    location_match
].copy()

print("Global events:", len(df_events))
print("European / strategic events:", len(df_europe))

print(
    "Share retained:",
    round(
        len(df_europe) / len(df_events) * 100,
        2
    ),
    "%"
)

Global events: 1175
European / strategic events: 267
Share retained: 22.72 %


## 10. CAMEO Event Classification

In [43]:
QUAD_CLASS_LABELS = {
    1: "Verbal Cooperation",
    2: "Material Cooperation",
    3: "Verbal Conflict",
    4: "Material Conflict"
}

df_europe["quad_class_label"] = (
    df_europe["quad_class"]
    .map(QUAD_CLASS_LABELS)
)

quad_distribution = (
    df_europe["quad_class_label"]
    .value_counts()
    .reset_index()
)

quad_distribution.columns = [
    "quad_class",
    "events"
]

quad_distribution

quad_distribution["share_pct"] = (
    quad_distribution["events"]
    / len(df_europe)
    * 100
).round(2)

quad_distribution

,quad_class,events,share_pct
0,Verbal Cooperation,189,70.79
1,Material Conflict,40,14.98
2,Material Cooperation,20,7.49
3,Verbal Conflict,18,6.74


In [44]:
df_europe["goldstein_scale"].describe()

count    267.000000
mean       0.870412
std        4.919799
min      -10.000000
25%        0.000000
50%        1.900000
75%        3.500000
max       10.000000
Name: goldstein_scale, dtype: float64

In [45]:
print(
    "Average Goldstein:",
    round(
        df_europe["goldstein_scale"].mean(),
        2
    )
)

print(
    "Minimum:",
    df_europe["goldstein_scale"].min()
)

print(
    "Maximum:",
    df_europe["goldstein_scale"].max()
)

Average Goldstein: 0.87
Minimum: -10.0
Maximum: 10.0


In [46]:
CAMEO_ROOT_CODES = {
    1: "Make Public Statement",
    2: "Appeal",
    3: "Express Intent to Cooperate",
    4: "Consult",
    5: "Engage in Diplomatic Cooperation",
    6: "Engage in Material Cooperation",
    7: "Provide Aid",
    8: "Yield",
    9: "Investigate",
    10: "Demand",
    11: "Disapprove",
    12: "Reject",
    13: "Threaten",
    14: "Protest",
    15: "Exhibit Force Posture",
    16: "Reduce Relations",
    17: "Coerce",
    18: "Assault",
    19: "Fight",
    20: "Use Unconventional Mass Violence"
}

In [47]:
df_europe["event_root_label"] = (
    df_europe["event_root_code"]
    .map(CAMEO_ROOT_CODES)
)

In [48]:
root_distribution = (
    df_europe["event_root_label"]
    .value_counts()
    .reset_index()
)

root_distribution.columns = [
    "event_type",
    "events"
]

root_distribution.head(15)

,event_type,events
0,Consult,85
1,Engage in Diplomatic Cooperation,31
2,Make Public Statement,27
3,Fight,27
4,Express Intent to Cooperate,23
5,Appeal,23
6,Coerce,9
7,Provide Aid,9
8,Disapprove,6
9,Investigate,6


## 11. Security Attention Classification

In [49]:
HIGH_ATTENTION_ROOTS = {
    13,  # Threaten
    15,  # Exhibit Force Posture
    16,  # Reduce Relations
    17,  # Coerce
}

CRITICAL_ATTENTION_ROOTS = {
    18,  # Assault
    19,  # Fight
    20   # Use Unconventional Mass Violence
}

MEDIUM_ATTENTION_ROOTS = {
    10,  # Demand
    11,  # Disapprove
    12,  # Reject
    14   # Protest
}



In [50]:
def classify_attention_level(row):
    
    root_code = row["event_root_code"]
    goldstein = row["goldstein_scale"]

    # Critical conflict events
    if root_code in CRITICAL_ATTENTION_ROOTS:
        return "Critical"

    # Strong conflict / coercive behaviour
    if root_code in HIGH_ATTENTION_ROOTS:
        return "High"

    # Moderate political tension
    if root_code in MEDIUM_ATTENTION_ROOTS:
        return "Medium"

    # Additional safeguard:
    # very negative Goldstein events deserve attention
    if pd.notna(goldstein) and goldstein <= -5:
        return "High"

    return "Low"

In [51]:
df_europe["attention_level"] = df_europe.apply(
    classify_attention_level,
    axis=1
)

In [52]:
attention_distribution = (
    df_europe["attention_level"]
    .value_counts()
    .reindex(
        ["Critical", "High", "Medium", "Low"],
        fill_value=0
    )
    .reset_index()
)

attention_distribution.columns = [
    "attention_level",
    "events"
]

attention_distribution["share_pct"] = (
    attention_distribution["events"]
    / len(df_europe)
    * 100
).round(2)

attention_distribution

,attention_level,events,share_pct
0,Critical,30,11.24
1,High,16,5.99
2,Medium,12,4.49
3,Low,209,78.28


In [53]:
df_europe[
    df_europe["attention_level"].isin(
        ["Critical", "High"]
    )
][
    [
        "event_date",
        "actor1",
        "actor2",
        "event_root_label",
        "quad_class_label",
        "goldstein_scale",
        "avg_tone",
        "location",
        "source_url"
    ]
].sort_values(
    "goldstein_scale"
).head(20)

,event_date,actor1,actor2,event_root_label,quad_class_label,goldstein_scale,avg_tone,location,source_url
74,2026-08-24,NaN,UNITED KINGDOM,Fight,Material Conflict,-10.0,-3.750837,"Cambridge, Cambridgeshire, United Kingdom",https://www.christiantoday.com/news/learning-f...
95,2026-08-24,NaN,ITALY,Fight,Material Conflict,-10.0,-7.301173,Ukraine,https://londonlovesbusiness.com/russias-shadow...
187,2026-08-24,BULGARIAN,NaN,Fight,Material Conflict,-10.0,-7.301173,"Kremlin, Moskva, Russia",https://londonlovesbusiness.com/russias-shadow...
194,2026-08-24,BRAZILIAN,LIMERICK,Fight,Material Conflict,-10.0,-3.030303,"Sorocaba, SãPaulo, Brazil",https://www.limerickpost.ie/2026/08/24/gardai-...
478,2026-08-24,FRANCE,RUSSIA,Fight,Material Conflict,-10.0,-1.324503,"Kremlin, Moskva, Russia",https://www.bangkokpost.com/world/3307108/krem...
477,2026-08-24,FRANCE,RUSSIA,Fight,Material Conflict,-10.0,-1.324503,"Kremlin, Moskva, Russia",https://www.bangkokpost.com/world/3307108/krem...
476,2026-08-24,FRANCE,RUSSIA,Fight,Material Conflict,-10.0,-1.000000,"Kyiv, Kyyiv, Misto, Ukraine",https://www.kyivpost.com/post/83001
347,2026-08-24,CRIMINAL,NaN,Fight,Material Conflict,-10.0,-7.301173,"Moscow, Moskva, Russia",https://londonlovesbusiness.com/russias-shadow...
479,2026-08-24,FRANCE,RUSSIA,Fight,Material Conflict,-10.0,-1.324503,United Kingdom,https://www.bangkokpost.com/world/3307108/krem...
508,2026-08-24,BRITAIN,RUSSIA,Fight,Material Conflict,-10.0,-1.194702,"Kremlin, Moskva, Russia",https://www.bangkokpost.com/world/3307108/krem...


In [54]:
attention_metrics = df_europe[
    [
        "num_mentions",
        "num_sources",
        "num_articles",
        "avg_tone"
    ]
].describe()

attention_metrics

,num_mentions,num_sources,num_articles,avg_tone
count,267.000000,267.000000,267.000000,267.000000
mean,4.101124,1.022472,3.921348,-1.407171
std,3.449022,0.192580,2.988291,3.158587
min,1.000000,1.000000,1.000000,-12.409812
25%,2.000000,1.000000,2.000000,-3.557312
50%,3.000000,1.000000,3.000000,-1.324503
75%,6.000000,1.000000,6.000000,0.755943
max,20.000000,3.000000,10.000000,5.494505


In [55]:
print("Correlation matrix:")

df_europe[
    [
        "num_mentions",
        "num_sources",
        "num_articles",
        "avg_tone",
        "goldstein_scale"
    ]
].corr().round(2)

Correlation matrix:


,num_mentions,num_sources,num_articles,avg_tone,goldstein_scale
num_mentions,1.00,0.06,0.95,0.17,-0.07
num_sources,0.06,1.00,0.07,0.16,-0.01
num_articles,0.95,0.07,1.00,0.16,-0.06
avg_tone,0.17,0.16,0.16,1.00,0.23
goldstein_scale,-0.07,-0.01,-0.06,0.23,1.00


## 12. Security Attention Score

In [56]:
CAMEO_SEVERITY = {
    1: 5,     # Make Public Statement
    2: 5,     # Appeal
    3: 5,     # Express Intent to Cooperate
    4: 5,     # Consult
    5: 5,     # Diplomatic Cooperation

    6: 10,    # Material Cooperation
    7: 10,    # Provide Aid
    8: 15,    # Yield

    9: 20,    # Investigate

    10: 35,   # Demand
    11: 35,   # Disapprove
    12: 40,   # Reject
    13: 60,   # Threaten
    14: 45,   # Protest
    15: 70,   # Exhibit Force Posture
    16: 65,   # Reduce Relations
    17: 75,   # Coerce

    18: 90,   # Assault
    19: 95,   # Fight
    20: 100   # Unconventional Mass Violence
}

df_europe["cameo_severity"] = (
    df_europe["event_root_code"]
    .map(CAMEO_SEVERITY)
    .fillna(0)
)

In [57]:
df_europe["goldstein_conflict"] = (
    (10 - df_europe["goldstein_scale"]) / 20 * 100
).clip(0, 100)

In [58]:
df_europe[
    [
        "goldstein_scale",
        "goldstein_conflict"
    ]
].head(10)

,goldstein_scale,goldstein_conflict
0,1.9,40.5
1,1.9,40.5
2,1.9,40.5
3,4.0,30.0
4,4.0,30.0
7,2.8,36.0
8,2.8,36.0
9,2.8,36.0
16,0.0,50.0
17,0.0,50.0


In [59]:
df_europe["event_severity_score"] = (
    0.60 * df_europe["cameo_severity"] +
    0.40 * df_europe["goldstein_conflict"]
)

In [60]:
import numpy as np

mentions_reference = df_europe["num_mentions"].quantile(0.95)

df_europe["media_attention_score"] = (
    np.log1p(df_europe["num_mentions"])
    / np.log1p(mentions_reference)
    * 100
).clip(0, 100)

In [61]:
print("95th percentile mentions:", mentions_reference)

df_europe[
    [
        "num_mentions",
        "media_attention_score"
    ]
].sort_values(
    "num_mentions",
    ascending=False
).head(10)

95th percentile mentions: 10.0


,num_mentions,media_attention_score
425,20,100.0
926,20,100.0
482,18,100.0
488,14,100.0
181,12,100.0
347,10,100.0
455,10,100.0
426,10,100.0
459,10,100.0
460,10,100.0


In [62]:
df_europe["negative_tone_score"] = (
    (-df_europe["avg_tone"]).clip(0, 10)
    / 10
    * 100
)

In [63]:
reference_date = df_europe["event_date"].max()

df_europe["days_old"] = (
    reference_date - df_europe["event_date"]
).dt.days

In [64]:
df_europe["recency_score"] = (
    np.exp(
        -df_europe["days_old"] / 14
    )
    * 100
)

In [65]:
df_europe["attention_score"] = (
    0.55 * df_europe["event_severity_score"] +
    0.20 * df_europe["media_attention_score"] +
    0.15 * df_europe["negative_tone_score"] +
    0.10 * df_europe["recency_score"]
).round(2)

In [66]:
def attention_band(score):

    if score >= 75:
        return "Critical"

    elif score >= 55:
        return "High"

    elif score >= 35:
        return "Medium"

    else:
        return "Low"


df_europe["attention_band"] = (
    df_europe["attention_score"]
    .apply(attention_band)
)

In [67]:
df_europe["attention_score"].describe()

count    267.000000
mean      41.482210
std       17.586509
min       20.730000
25%       30.135000
50%       34.770000
75%       45.320000
max       94.300000
Name: attention_score, dtype: float64

In [68]:
attention_score_distribution = (
    df_europe["attention_band"]
    .value_counts()
    .reindex(
        ["Critical", "High", "Medium", "Low"],
        fill_value=0
    )
)

attention_score_distribution

attention_band
Critical     23
High         28
Medium       81
Low         135
Name: count, dtype: int64

In [69]:
top_attention_events = (
    df_europe[
        [
            "event_date",
            "actor1",
            "actor2",
            "event_root_label",
            "quad_class_label",
            "goldstein_scale",
            "num_mentions",
            "avg_tone",
            "location",
            "attention_score",
            "attention_band",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .head(20)
)

top_attention_events

,event_date,actor1,actor2,event_root_label,quad_class_label,goldstein_scale,num_mentions,avg_tone,location,attention_score,attention_band,source_url
95,2026-08-24,NaN,ITALY,Fight,Material Conflict,-10.0,10,-7.301173,Ukraine,94.30,Critical,https://londonlovesbusiness.com/russias-shadow...
347,2026-08-24,CRIMINAL,NaN,Fight,Material Conflict,-10.0,10,-7.301173,"Moscow, Moskva, Russia",94.30,Critical,https://londonlovesbusiness.com/russias-shadow...
194,2026-08-24,BRAZILIAN,LIMERICK,Fight,Material Conflict,-10.0,10,-3.030303,"Sorocaba, SãPaulo, Brazil",87.90,Critical,https://www.limerickpost.ie/2026/08/24/gardai-...
489,2026-08-24,UNITED KINGDOM,NaN,Assault,Material Conflict,-9.0,8,-5.065666,"Hemel Hempstead, Hertfordshire, United Kingdom",86.52,Critical,https://www.hertsad.co.uk/news/26489327.herts-...
492,2026-08-24,GLASGOW,NaN,Fight,Material Conflict,-10.0,10,-2.048417,"Glasgow, Glasgow City, United Kingdom",86.42,Critical,https://www.dailyrecord.co.uk/news/scottish-ne...
1058,2026-08-24,UKRAINIAN,RUSSIAN,Fight,Material Conflict,-10.0,6,-3.866667,"Kyiv, Kyyiv, Misto, Ukraine",85.38,Critical,https://www.kyivpost.com/opinion/82914
491,2026-08-24,EDINBURGH,NaN,Fight,Material Conflict,-10.0,6,-3.769841,"Edinburgh, Edinburgh, City of, United Kingdom",85.23,Critical,https://www.dailyrecord.co.uk/tv/mesmerising-b...
980,2026-08-24,RUSSIAN,UKRAINE,Fight,Material Conflict,-10.0,6,-3.557312,"Vede, Russia (general), Russia",84.92,Critical,https://www.kyivpost.com/post/82985
839,2026-08-24,MOLDOVAN,ROMANIA,Fight,Material Conflict,-9.5,6,-3.557312,"Vede, Russia (general), Russia",84.37,Critical,https://www.kyivpost.com/post/82985
187,2026-08-24,BULGARIAN,NaN,Fight,Material Conflict,-10.0,2,-7.301173,"Kremlin, Moskva, Russia",83.46,Critical,https://londonlovesbusiness.com/russias-shadow...


In [70]:
print(
    "Top 20 events:",
    len(top_attention_events)
)

print(
    "Unique source URLs:",
    top_attention_events["source_url"].nunique()
)

Top 20 events: 20
Unique source URLs: 10


In [71]:
source_frequency = (
    df_europe
    .groupby("source_url")
    .agg(
        event_count=("event_id", "count"),
        max_attention_score=("attention_score", "max"),
        avg_attention_score=("attention_score", "mean")
    )
    .sort_values(
        "event_count",
        ascending=False
    )
)

source_frequency.head(15)

,event_count,max_attention_score,avg_attention_score
source_url,,,
https://gcaptain.com/finnish-icebreaker-nordica-departs-for-canada-in-support-of-arctic-mission/,22,33.19,28.644545
https://www.bangkokpost.com/world/3307108/kremlin-says-uk-preventing-peace-by-giving-missile-info-to-ukraine,19,80.09,42.223158
https://londonlovesbusiness.com/kremlin-warns-britain-over-burnhams-secret-storm-shadow-plans/,14,60.36,44.754286
https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/,12,41.37,31.218333
https://www.kyivpost.com/post/82985,11,84.92,61.418182
https://english.radio.cz/costa-visit-prague-thursday-part-tour-des-capitales-8895438,11,40.56,33.221818
https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/,10,94.30,74.416000
https://www.hertsad.co.uk/news/26489327.herts-teacher-accused-sex-pupils-was-just-friendly/,10,86.52,46.419000
https://www.theportugalnews.com/news/2026-08-24/tame-impala-return-to-portugal-in-2027/1074630,9,39.57,26.850000


In [72]:
top_unique_articles = (
    df_europe
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    [
        [
            "event_date",
            "actor1",
            "actor2",
            "event_root_label",
            "goldstein_scale",
            "location",
            "attention_score",
            "attention_band",
            "source_url"
        ]
    ]
    .head(20)
)

top_unique_articles

,event_date,actor1,actor2,event_root_label,goldstein_scale,location,attention_score,attention_band,source_url
95,2026-08-24,NaN,ITALY,Fight,-10.0,Ukraine,94.30,Critical,https://londonlovesbusiness.com/russias-shadow...
194,2026-08-24,BRAZILIAN,LIMERICK,Fight,-10.0,"Sorocaba, SãPaulo, Brazil",87.90,Critical,https://www.limerickpost.ie/2026/08/24/gardai-...
489,2026-08-24,UNITED KINGDOM,NaN,Assault,-9.0,"Hemel Hempstead, Hertfordshire, United Kingdom",86.52,Critical,https://www.hertsad.co.uk/news/26489327.herts-...
492,2026-08-24,GLASGOW,NaN,Fight,-10.0,"Glasgow, Glasgow City, United Kingdom",86.42,Critical,https://www.dailyrecord.co.uk/news/scottish-ne...
1058,2026-08-24,UKRAINIAN,RUSSIAN,Fight,-10.0,"Kyiv, Kyyiv, Misto, Ukraine",85.38,Critical,https://www.kyivpost.com/opinion/82914
491,2026-08-24,EDINBURGH,NaN,Fight,-10.0,"Edinburgh, Edinburgh, City of, United Kingdom",85.23,Critical,https://www.dailyrecord.co.uk/tv/mesmerising-b...
980,2026-08-24,RUSSIAN,UKRAINE,Fight,-10.0,"Vede, Russia (general), Russia",84.92,Critical,https://www.kyivpost.com/post/82985
74,2026-08-24,NaN,UNITED KINGDOM,Fight,-10.0,"Cambridge, Cambridgeshire, United Kingdom",82.40,Critical,https://www.christiantoday.com/news/learning-f...
508,2026-08-24,BRITAIN,RUSSIA,Fight,-10.0,"Kremlin, Moskva, Russia",80.09,Critical,https://www.bangkokpost.com/world/3307108/krem...
552,2026-08-24,FIRE COMPANY,NaN,Fight,-10.0,"Smyrna, Izmir, Turkey",78.29,Critical,https://www.eaglecountryonline.com/news/local-...


In [73]:
lines = response.text.strip().splitlines()

gkg_line = lines[2]

gkg_url = gkg_line.split()[-1]

gkg_url = gkg_url.replace(
    "http://",
    "https://"
)

print("Latest GDELT GKG file:")
print(gkg_url)

Latest GDELT GKG file:
https://data.gdeltproject.org/gdeltv2/20260824131500.gkg.csv.zip


In [74]:
gkg_file_name = gkg_url.split("/")[-1]

gkg_zip_path = (
    raw_dir
    / gkg_file_name
)

gkg_response = requests.get(
    gkg_url,
    timeout=60
)

gkg_response.raise_for_status()

with open(
    gkg_zip_path,
    "wb"
) as file:
    
    file.write(
        gkg_response.content
    )

print("Downloaded:")
print(gkg_zip_path)

print(
    "File size:",
    round(
        gkg_zip_path.stat().st_size
        / 1024,
        2
    ),
    "KB"
)

Downloaded:
c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\raw\20260824131500.gkg.csv.zip
File size: 4670.92 KB


## 13. Load GDELT GKG

In [75]:
GKG_COLUMNS = [
    "GKGRECORDID",
    "V2DATE",
    "V2SOURCECOLLECTIONIDENTIFIER",
    "V2SOURCECOMMONNAME",
    "V2DOCUMENTIDENTIFIER",
    "V1COUNTS",
    "V2COUNTS",
    "V1THEMES",
    "V2ENHANCEDTHEMES",
    "V1LOCATIONS",
    "V2ENHANCEDLOCATIONS",
    "V1PERSONS",
    "V2ENHANCEDPERSONS",
    "V1ORGANIZATIONS",
    "V2ENHANCEDORGANIZATIONS",
    "V1TONE",
    "V2ENHANCEDDATES",
    "V2GCAM",
    "V2SHARINGIMAGE",
    "V2RELATEDIMAGES",
    "V2SOCIALIMAGEEMBEDS",
    "V2SOCIALVIDEOEMBEDS",
    "V2QUOTATIONS",
    "V2ALLNAMES",
    "V2AMOUNTS",
    "V2TRANSLATIONINFO",
    "V2EXTRASXML"
]

In [76]:
df_gkg = pd.read_csv(
    gkg_zip_path,
    sep="\t",
    header=None,
    names=GKG_COLUMNS,
    compression="zip",
    low_memory=False
)

print("Rows:", len(df_gkg))
print("Columns:", len(df_gkg.columns))

df_gkg.head()

Rows: 1151
Columns: 27


,GKGRECORDID,V2DATE,V2SOURCECOLLECTIONIDENTIFIER,V2SOURCECOMMONNAME,V2DOCUMENTIDENTIFIER,V1COUNTS,V2COUNTS,V1THEMES,V2ENHANCEDTHEMES,V1LOCATIONS,...,V2GCAM,V2SHARINGIMAGE,V2RELATEDIMAGES,V2SOCIALIMAGEEMBEDS,V2SOCIALVIDEOEMBEDS,V2QUOTATIONS,V2ALLNAMES,V2AMOUNTS,V2TRANSLATIONINFO,V2EXTRASXML
0,20260824131500-0,20260824131500,1,bangladeshsun.com,http://www.bangladeshsun.com/news/279261963/fo...,NaN,NaN,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_MAHARASH...,"WB_1467_EDUCATION_FOR_ALL,837;WB_470_EDUCATION...","4#Mumbai, Maharashtra, India#IN#IN16#18.975#72...",...,"wc:854,c1.3:2,c12.1:40,c12.10:79,c12.11:1,c12....",https://image.chitra.live/api/v1/wps/70c85e2/e...,NaN,NaN,https://youtube.com/embed/R3826ZVTKmE;https://...,216|46||Point of Care Ultrasound in Emergency ...,"Fortis Hospital,62;Care Ultrasound,199;Fortis ...","450,emergency medicine doctors across,1362;2,e...",NaN,<PAGE_LINKS>http://www.fortishealthcare.com/;h...
1,20260824131500-1,20260824131500,1,theconversation.com,https://theconversation.com/fish-dna-in-ozark-...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_STUDENTS;UNGP_FORESTS_RI...,"TAX_WORLDFISH_SUNFISHES,4113;SCIENCE,828;TAX_W...","2#Alabama, United States#US#USAL#32.799#-86.80...",...,"wc:671,c1.1:1,c1.3:7,c12.1:29,c12.10:54,c12.12...",https://images.theconversation.com/files/75490...,NaN,NaN,https://youtube.com/@TheConversationUS;https:/...,NaN,"Missouri Department,90;Northern Ozark,1767;Bla...","12,rivers,164;100000,of thousands of years,1865;",NaN,<PAGE_LINKS>https://archive.org/details/fishes...
2,20260824131500-2,20260824131500,1,actuarialpost.co.uk,https://www.actuarialpost.co.uk/article/two-th...,NaN,NaN,RETIREMENT;TAX_FNCACT;TAX_FNCACT_DRIVER;WB_269...,"TAX_FNCACT_DRIVER,312;TAX_FNCACT_DRIVER,1526;T...",1#United Kingdom#UK#UK#54#-4#UK,...,"wc:700,c1.3:1,c12.1:41,c12.10:71,c12.12:22,c12...",NaN,NaN,NaN,NaN,NaN,"Customer Savings,3282;Investment Director,3304...","2,fifths,315;3,quarters,577;2,thirds of the to...",NaN,<PAGE_LINKS>https://www.actuarialpost.co.uk/ar...
3,20260824131500-3,20260824131500,1,livelaw.in,https://www.livelaw.in/news-updates/karnataka-...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_CHAIRMAN;RESIGNATION;EPU...,"EPU_POLICY_REGULATORY,918;RESIGNATION,695;ELEC...","1#India#IN#IN#20#77#IN;5#Karnataka, Karnataka,...",...,"wc:311,c12.1:28,c12.10:33,c12.11:1,c12.12:16,c...",https://www.livelaw.in/h-upload/2025/03/05/589...,NaN,NaN,NaN,640|281||The KSBC firmly dismisses and condemn...,"Senior Advocate Y R Sadashiva Reddy,76;Bar Cou...",NaN,NaN,<PAGE_LINKS>https://www.livelaw.in/top-stories...
4,20260824131500-4,20260824131500,1,siliconangle.com,https://siliconangle.com/2026/08/24/thomson-re...,NaN,NaN,TAX_FNCACT;TAX_FNCACT_ASSISTANT;TAX_FNCACT_ADM...,"TAX_FNCACT_ADMINISTRATORS,622;CRISISLEX_CRISIS...",NaN,...,"wc:746,c1.2:1,c12.1:38,c12.10:84,c12.12:21,c12...",NaN,NaN,NaN,NaN,NaN,"Thomson Reuters,42;Tabular Analysis,297;CoCoun...","450,dollars ,708;100,of subject,1628;400,tech,...",NaN,<PAGE_LINKS>https://flic.kr/p/oMa9mU;https://l...


In [77]:
print(df_gkg.columns.tolist())

['GKGRECORDID', 'V2DATE', 'V2SOURCECOLLECTIONIDENTIFIER', 'V2SOURCECOMMONNAME', 'V2DOCUMENTIDENTIFIER', 'V1COUNTS', 'V2COUNTS', 'V1THEMES', 'V2ENHANCEDTHEMES', 'V1LOCATIONS', 'V2ENHANCEDLOCATIONS', 'V1PERSONS', 'V2ENHANCEDPERSONS', 'V1ORGANIZATIONS', 'V2ENHANCEDORGANIZATIONS', 'V1TONE', 'V2ENHANCEDDATES', 'V2GCAM', 'V2SHARINGIMAGE', 'V2RELATEDIMAGES', 'V2SOCIALIMAGEEMBEDS', 'V2SOCIALVIDEOEMBEDS', 'V2QUOTATIONS', 'V2ALLNAMES', 'V2AMOUNTS', 'V2TRANSLATIONINFO', 'V2EXTRASXML']


In [78]:
df_gkg_core = df_gkg[
    [
        "GKGRECORDID",
        "V2DATE",
        "V2SOURCECOMMONNAME",
        "V2DOCUMENTIDENTIFIER",
        "V1THEMES",
        "V2ENHANCEDTHEMES",
        "V1LOCATIONS",
        "V1ORGANIZATIONS",
        "V1TONE"
    ]
].copy()

In [79]:
df_gkg_core = df_gkg_core.rename(columns={
    "GKGRECORDID": "gkg_record_id",
    "V2DATE": "gkg_date",
    "V2SOURCECOMMONNAME": "source_name",
    "V2DOCUMENTIDENTIFIER": "document_url",
    "V1THEMES": "themes",
    "V2ENHANCEDTHEMES": "enhanced_themes",
    "V1LOCATIONS": "locations",
    "V1ORGANIZATIONS": "organizations",
    "V1TONE": "tone_raw"
})

In [80]:
df_gkg_core[
    [
        "document_url",
        "source_name",
        "themes",
        "enhanced_themes"
    ]
].head(10)

,document_url,source_name,themes,enhanced_themes
0,http://www.bangladeshsun.com/news/279261963/fo...,bangladeshsun.com,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_MAHARASH...,"WB_1467_EDUCATION_FOR_ALL,837;WB_470_EDUCATION..."
1,https://theconversation.com/fish-dna-in-ozark-...,theconversation.com,TAX_FNCACT;TAX_FNCACT_STUDENTS;UNGP_FORESTS_RI...,"TAX_WORLDFISH_SUNFISHES,4113;SCIENCE,828;TAX_W..."
2,https://www.actuarialpost.co.uk/article/two-th...,actuarialpost.co.uk,RETIREMENT;TAX_FNCACT;TAX_FNCACT_DRIVER;WB_269...,"TAX_FNCACT_DRIVER,312;TAX_FNCACT_DRIVER,1526;T..."
3,https://www.livelaw.in/news-updates/karnataka-...,livelaw.in,TAX_FNCACT;TAX_FNCACT_CHAIRMAN;RESIGNATION;EPU...,"EPU_POLICY_REGULATORY,918;RESIGNATION,695;ELEC..."
4,https://siliconangle.com/2026/08/24/thomson-re...,siliconangle.com,TAX_FNCACT;TAX_FNCACT_ASSISTANT;TAX_FNCACT_ADM...,"TAX_FNCACT_ADMINISTRATORS,622;CRISISLEX_CRISIS..."
5,https://www.rhyljournal.co.uk/news/national/26...,rhyljournal.co.uk,ECON_WORLDCURRENCIES;ECON_WORLDCURRENCIES_DOLL...,"EPU_ECONOMY,641;EPU_ECONOMY,1226;EPU_ECONOMY,2..."
6,https://www.kmbc.com/article/phoenix-police-of...,kmbc.com,SECURITY_SERVICES;TAX_FNCACT;TAX_FNCACT_POLICE...,"TAX_FNCACT_POLICE_CHIEF,355;TAX_FNCACT_POLICE_..."
7,https://www.womenshealthmag.com/fitness/a73484...,womenshealthmag.com,MARITIME_INCIDENT;MARITIME;MANMADE_DISASTER_IM...,"MARITIME_INCIDENT,443;MARITIME_INCIDENT,1209;M..."
8,https://1057thehawk.com/ixp/397/p/asads-hot-ch...,1057thehawk.com,WB_678_DIGITAL_GOVERNMENT;WB_694_BROADCAST_AND...,"CRISISLEX_O01_WEATHER,1951;CRISISLEX_CRISISLEX..."
9,https://www.qatar-tribune.com/article/250277/l...,qatar-tribune.com,ECON_ENTREPRENEURSHIP;,"ECON_ENTREPRENEURSHIP,137;ECON_ENTREPRENEURSHI..."


In [81]:
matched_urls = df_europe["source_url"].isin(
    df_gkg_core["document_url"]
)

print("European events:", len(df_europe))
print("Events with GKG match:", matched_urls.sum())

print(
    "Match rate:",
    round(
        matched_urls.mean() * 100,
        2
    ),
    "%"
)

European events: 267
Events with GKG match: 267
Match rate: 100.0 %


## 14. Validate GKG Join

In [82]:
print("GKG rows:", len(df_gkg_core))

print(
    "Unique document URLs:",
    df_gkg_core["document_url"].nunique()
)

print(
    "Duplicated document URLs:",
    df_gkg_core["document_url"].duplicated().sum()
)

GKG rows: 1151
Unique document URLs: 1151
Duplicated document URLs: 0


In [83]:
df_security = df_europe.merge(
    df_gkg_core,
    left_on="source_url",
    right_on="document_url",
    how="left",
    validate="many_to_one"
)

In [84]:
print("Before merge:", len(df_europe))
print("After merge:", len(df_security))

print(
    "Missing GKG themes:",
    df_security["themes"].isna().sum()
)

Before merge: 267
After merge: 267
Missing GKG themes: 1


## 15. GKG Theme Inspection

In [85]:
df_articles = (
    df_gkg_core[
        df_gkg_core["document_url"].isin(
            df_europe["source_url"]
        )
    ]
    .copy()
)

print("European / strategic events:", len(df_europe))
print("Unique related articles:", len(df_articles))

European / strategic events: 267
Unique related articles: 60


In [86]:
def split_gkg_themes(value):
    """
    Converts the semicolon-separated GKG theme field
    into a Python list.
    """

    if pd.isna(value):
        return []

    return [
        theme.strip()
        for theme in str(value).split(";")
        if theme.strip()
    ]


df_articles["theme_list"] = (
    df_articles["themes"]
    .apply(split_gkg_themes)
)

In [87]:
df_articles[
    [
        "source_name",
        "document_url",
        "theme_list"
    ]
].head(10)

,source_name,document_url,theme_list
19,indiatimes.com,https://timesofindia.indiatimes.com/india/pays...,"[TAX_ETHNICITY, TAX_ETHNICITY_RUSSIAN, TAX_WOR..."
31,ummid.com,https://www.ummid.com/news/2026/8/24/managed-p...,"[CONSTITUTIONAL, TAX_ETHNICITY, TAX_ETHNICITY_..."
34,irishconstruction.com,https://irishconstruction.com/constructing-equ...,"[WB_615_GENDER, MOVEMENT_WOMENS, WB_919_GENDER..."
55,ibtimes.co.uk,https://www.ibtimes.co.uk/epomaker-he75-v2-fut...,[]
56,mining-technology.com,https://www.mining-technology.com/news/greenla...,"[ENV_MINING, GENERAL_GOVERNMENT, EPU_POLICY, E..."
68,english.radio.cz,https://english.radio.cz/foreign-minister-russ...,"[TAX_WORLDLANGUAGES, TAX_WORLDLANGUAGES_RUSSIA..."
73,gcaptain.com,https://gcaptain.com/finnish-icebreaker-nordic...,"[TAX_ETHNICITY, TAX_ETHNICITY_CANADIAN, TAX_FN..."
74,dailyrecord.co.uk,https://www.dailyrecord.co.uk/tv/mesmerising-b...,"[RURAL, CRISISLEX_CRISISLEXREC, SOC_GENERALCRI..."
82,kyivpost.com,https://www.kyivpost.com/post/82990,"[MILITARY_COOPERATION, SLFID_MILITARY_SPENDING..."
116,tribune.com.pk,https://tribune.com.pk/story/2625563/as-ai-age...,"[TAX_FNCACT, TAX_FNCACT_AGENTS, CYBER_ATTACK, ..."


In [88]:
theme_frequency = (
    df_articles["theme_list"]
    .explode()
    .dropna()
    .value_counts()
    .reset_index()
)

theme_frequency.columns = [
    "theme",
    "article_count"
]

theme_frequency.head(50)

,theme,article_count
0,TAX_FNCACT,56
1,TAX_ETHNICITY,37
2,EPU_POLICY,33
3,TAX_WORLDLANGUAGES,32
4,GENERAL_GOVERNMENT,29
5,WB_696_PUBLIC_SECTOR_MANAGEMENT,28
6,USPEC_POLITICS_GENERAL1,27
7,LEADER,26
8,CRISISLEX_CRISISLEXREC,25
9,CRISISLEX_C07_SAFETY,24


In [89]:
sample_articles = (
    df_security[
        [
            "source_name",
            "document_url",
            "event_root_label",
            "location",
            "attention_score",
            "themes"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="document_url"
    )
    .head(15)
)

sample_articles

,source_name,document_url,event_root_label,location,attention_score,themes
21,londonlovesbusiness.com,https://londonlovesbusiness.com/russias-shadow...,Fight,Ukraine,94.30,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_RUSSIA;R...
43,limerickpost.ie,https://www.limerickpost.ie/2026/08/24/gardai-...,Fight,"Sorocaba, SãPaulo, Brazil",87.90,CRISISLEX_CRISISLEXREC;CRISISLEX_T03_DEAD;TAX_...
117,hertsad.co.uk,https://www.hertsad.co.uk/news/26489327.herts-...,Assault,"Hemel Hempstead, Hertfordshire, United Kingdom",86.52,EDUCATION;SOC_POINTSOFINTEREST;SOC_POINTSOFINT...
120,dailyrecord.co.uk,https://www.dailyrecord.co.uk/news/scottish-ne...,Fight,"Glasgow, Glasgow City, United Kingdom",86.42,SOC_GENERALCRIME;CRISISLEX_C07_SAFETY;SEIZE;TA...
258,kyivpost.com,https://www.kyivpost.com/opinion/82914,Fight,"Kyiv, Kyyiv, Misto, Ukraine",85.38,TAX_ETHNICITY;TAX_ETHNICITY_UKRAINIANS;TAX_WOR...
119,dailyrecord.co.uk,https://www.dailyrecord.co.uk/tv/mesmerising-b...,Fight,"Edinburgh, Edinburgh, City of, United Kingdom",85.23,RURAL;CRISISLEX_CRISISLEXREC;SOC_GENERALCRIME;...
234,kyivpost.com,https://www.kyivpost.com/post/82985,Fight,"Vede, Russia (general), Russia",84.92,DRONES;TAX_ETHNICITY;TAX_ETHNICITY_RUSSIAN;TAX...
19,christiantoday.com,https://www.christiantoday.com/news/learning-f...,Fight,"Cambridge, Cambridgeshire, United Kingdom",82.40,TAX_RELIGION;TAX_RELIGION_CHRISTIAN;TAX_ETHNIC...
136,bangkokpost.com,https://www.bangkokpost.com/world/3307108/krem...,Fight,"Kremlin, Moskva, Russia",80.09,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_RUSSIA;T...
153,eaglecountryonline.com,https://www.eaglecountryonline.com/news/local-...,Fight,"Smyrna, Izmir, Turkey",78.29,TAX_FNCACT;TAX_FNCACT_VOLUNTEER;TAX_FNCACT_ASS...


## 16. Security Theme Discovery

In [90]:
security_search_terms = [
    "ARMED",
    "MILITARY",
    "DEFEN",
    "WEAPON",
    "MISSILE",
    "WAR",
    "CONFLICT",
    "CYBER",
    "HACK",
    "SANCTION",
    "ENERGY",
    "OIL",
    "GAS",
    "NUCLEAR",
    "TERROR",
    "SECURITY",
    "NATO",
    "RUSSIA",
    "UKRAINE"
]

In [91]:
security_theme_candidates = theme_frequency[
    theme_frequency["theme"]
    .str.contains(
        "|".join(security_search_terms),
        case=False,
        na=False
    )
].copy()

security_theme_candidates.head(100)

,theme,article_count
13,WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE,21
14,ARMEDCONFLICT,21
16,EPU_CATS_NATIONAL_SECURITY,20
21,TAX_WORLDLANGUAGES_RUSSIA,17
27,WB_2433_CONFLICT_AND_VIOLENCE,14
...,...,...
642,SOC_POINTSOFINTEREST_MILITARY_BASES,1
670,TAX_FNCACT_WARRIORS,1
693,TAX_TERROR_GROUP_HAMAS,1
696,TAX_MILITARY_TITLE_COMMANDER,1


In [92]:
def find_themes(term):
    return theme_frequency[
        theme_frequency["theme"]
        .str.contains(
            term,
            case=False,
            na=False
        )
    ]

In [93]:
find_themes("ARMED")

,theme,article_count
14,ARMEDCONFLICT,21


In [94]:
find_themes("MILITARY")

,theme,article_count
111,TAX_MILITARY_TITLE,5
159,MILITARY,4
176,MILITARY_COOPERATION,3
263,TAX_MILITARY_TITLE_OFFICER,2
309,TAX_MILITARY_TITLE_OFFICERS,2
314,TAX_MILITARY_TITLE_SUPERINTENDENT,2
387,SLFID_MILITARY_SPENDING,1
493,TAX_FNCACT_MILITARY_LEADERS,1
642,SOC_POINTSOFINTEREST_MILITARY_BASES,1
696,TAX_MILITARY_TITLE_COMMANDER,1


In [95]:
find_themes("CYBER")

,theme,article_count
168,CYBER_ATTACK,3


In [96]:
find_themes("SANCTION")

,theme,article_count
231,SANCTIONS,2


In [97]:
find_themes("ENERGY")

,theme,article_count
49,WB_507_ENERGY_AND_EXTRACTIVES,9
126,WB_509_NUCLEAR_ENERGY,4


In [98]:
find_themes("CONFLICT")

,theme,article_count
13,WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE,21
14,ARMEDCONFLICT,21
27,WB_2433_CONFLICT_AND_VIOLENCE,14
32,WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT,12
455,WB_2480_POST_CONFLICT_RECONSTRUCTION,1


In [99]:
pd.set_option("display.max_colwidth", None)

In [100]:
security_samples = (
    df_security
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="document_url"
    )
    [
        [
            "document_url",
            "event_root_label",
            "location",
            "attention_score",
            "themes"
        ]
    ]
    .head(10)
)

security_samples

,document_url,event_root_label,location,attention_score,themes
21,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/,Fight,Ukraine,94.30,TAX_WORLDLANGUAGES;TAX_WORLDLANGUAGES_RUSSIA;RECRUITMENT;TAX_FNCACT;TAX_FNCACT_CRIMINAL;CRM_ARSON;CRISISLEX_C07_SAFETY;LEGISLATION;EPU_POLICY;EPU_POLICY_LAW;CRISISLEX_CRISISLEXREC;SECURITY_SERVICES;WB_696_PUBLIC_SECTOR_MANAGEMENT;WB_840_JUSTICE;WB_1920_FINANCIAL_SECTOR_DEVELOPMENT;WB_328_FINANCIAL_INTEGRITY;WB_1014_CRIMINAL_JUSTICE;WB_2082_LAW_ENFORCEMENT;TAX_FNCACT_OFFICIALS;ALLIANCE;TAX_FNCACT_OFFICIAL;CYBER_ATTACK;MANMADE_DISASTER_IMPLIED;TAX_ETHNICITY;TAX_ETHNICITY_ESTONIAN;TAX_WORLDLANGUAGES_ESTONIAN;DRONES;TAX_FNCACT_MANUFACTURER;TAX_ETHNICITY_LATVIAN;TAX_WORLDLANGUAGES_LATVIAN;TAX_FNCACT_AUTHORITIES;EPU_POLICY_AUTHORITIES;ARREST;SOC_GENERALCRIME;TAX_FNCACT_CITIZENS;TAX_FNCACT_MINISTER;LEADER;TAX_FNCACT_PRIME_MINISTER;DISASTER_FIRE;CRISISLEX_T01_CAUTION_ADVICE;WB_135_TRANSPORT;WB_1174_WAREHOUSING_AND_STORAGE;WB_793_TRANSPORT_AND_LOGISTICS_SERVICES;TAX_ETHNICITY_BULGARIAN;TAX_WORLDLANGUAGES_BULGARIAN;TAX_WEAPONS;TAX_WEAPONS_ARTILLERY;TAX_ETHNICITY_CZECH;TAX_WORLDLANGUAGES_CZECH;GENERAL_GOVERNMENT;TAX_ETHNICITY_LITHUANIAN;TAX_WORLDLANGUAGES_LITHUANIAN;TRIAL;WB_1921_PRIVATE_SECTOR_DEVELOPMENT;WB_346_COMPETITIVE_INDUSTRIES;WB_818_INDUSTRY_POLICY_AND_REAL_SECTORS;WB_1281_MANUFACTURING;TAX_ETHNICITY_UKRAINIAN;TAX_WORLDLANGUAGES_UKRAINIAN;EPU_ECONOMY_HISTORIC;MANMADE_DISASTER;MANMADE_DISASTER_INDUSTRIAL_ACCIDENT;EMERG_INDUSTRIALACCIDENT;WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT;WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE;WB_2490_NATIONAL_PROTECTION_AND_SECURITY;WB_1934_CIVILLIAN_POLICE_AND_SECURITY_SERVICES;TAX_ETHNICITY_RUSSIAN;TAX_WORLDLANGUAGES_RUSSIAN;TAX_MILITARY_TITLE;TAX_MILITARY_TITLE_OFFICERS;TAX_FNCACT_OFFICERS;ECON_BITCOIN;TAX_MILITARY_TITLE_OFFICER;TAX_FNCACT_OFFICER;TAX_FNCACT_INTELLIGENCE_OFFICER;EPU_POLICY_POLITICAL;TAX_FNCACT_CRIMINALS;TAX_ECON_PRICE;TAX_DISEASE;TAX_DISEASE_CONVENTIONAL;ARMEDCONFLICT;EPU_CATS_NATIONAL_SECURITY;WB_2468_CONVENTIONAL_WAR;WB_2433_CONFLICT_AND_VIOLENCE;WB_2462_POLITICAL_VIOLENCE_AND_WAR;
43,https://www.limerickpost.ie/2026/08/24/gardai-treating-death-of-brazilian-woman-in-limerick-as-murder-suicide-after-suspects-body-found/,Fight,"Sorocaba, SãPaulo, Brazil",87.90,CRISISLEX_CRISISLEXREC;CRISISLEX_T03_DEAD;TAX_FNCACT;TAX_FNCACT_CHIEF;KILL;CRISISLEX_T02_INJURED;TAX_FNCACT_SPOKESMAN;WB_2024_ANTI_CORRUPTION_AUTHORITIES;WB_696_PUBLIC_SECTOR_MANAGEMENT;WB_840_JUSTICE;WB_2025_INVESTIGATION;WB_831_GOVERNANCE;WB_832_ANTI_CORRUPTION;WB_1014_CRIMINAL_JUSTICE;SOC_GENERALCRIME;TAX_FNCACT_WOMAN;TAX_MILITARY_TITLE;TAX_MILITARY_TITLE_OFFICER;TAX_FNCACT_OFFICER;APPOINTMENT;MANMADE_DISASTER_IMPLIED;STRIKE;TAX_FNCACT_PARAMEDICS;WB_1428_INJURY;WB_1406_DISEASES;WB_621_HEALTH_NUTRITION_AND_POPULATION;WB_1427_NON_COMMUNICABLE_DISEASE_AND_INJURY;TAX_FNCACT_GUARD;SOC_USSECURITYAGENCIES;TAX_ECON_PRICE;CRISISLEX_T11_UPDATESSYMPATHY;TAX_DISEASE;TAX_DISEASE_CONTAGIOUS;
117,https://www.hertsad.co.uk/news/26489327.herts-teacher-accused-sex-pupils-was-just-friendly/,Assault,"Hemel Hempstead, Hertfordshire, United Kingdom",86.52,EDUCATION;SOC_POINTSOFINTEREST;SOC_POINTSOFINTEREST_COLLEGE;SOC_POINTSOFINTEREST_SCHOOL;TAX_FNCACT;TAX_FNCACT_TEACHER;TAX_FNCACT_STUDENTS;UNGP_CRIME_VIOLENCE;TAX_FNCACT_VICTIM;GENERAL_HEALTH;MEDICAL;CRISISLEX_C03_WELLBEING_HEALTH;TAX_ETHNICITY;TAX_ETHNICITY_BLACK;TAX_FNCACT_CHILDREN;HARASSMENT;TRIAL;CRISISLEX_T11_UPDATESSYMPATHY;TAX_FNCACT_JUDGE;
120,https://www.dailyrecord.co.uk/news/scottish-news/glasgow-murder-cops-seize-4000-37586658,Fight,"Glasgow, Glasgow City, United Kingdom",86.42,SOC_GENERALCRIME;CRISISLEX_C07_SAFETY;SEIZE;TAX_FNCACT;TAX_FNCACT_KILLER;WOUND;CRISISLEX_CRISISLEXREC;CRISISLEX_C03_WELLBEING_HEALTH;CRISISLEX_T02_INJURED;SECURITY_SERVICES;TAX_FNCACT_POLICE;KILL;CRISISLEX_T03_DEAD;TAX_FNCACT_CHANCELLOR;TAX_FNCACT_SPECIALIST;TAX_MILITARY_TITLE;TAX_MILITARY_TITLE_OFFICERS;TAX_FNCACT_OFFICERS;TAX_FNCAC

## 17. Security Relevance Layer

In [101]:
GKG_SECURITY_THEMES = {

    "Defence & Military": {
        "MILITARY",
        "MILITARY_COOPERATION",
        "TAX_WEAPONS_DRONE_STRIKE",
        "TAX_WEAPONS_ARTILLERY",
        "ARMEDCONFLICT",
        "WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT"
    },

    "Cybersecurity": {
        "CYBER_ATTACK",
        "WB_670_ICT_SECURITY",
        "TAX_FNCACT_HACKER",
        "TAX_FNCACT_HACKERS"
    },

    "Energy Security": {
        "WB_507_ENERGY_AND_EXTRACTIVES",
        "ENV_OIL",
        "ENV_NATURALGAS",
        "WB_539_OIL_AND_GAS_POLICY_STRATEGY_AND_INSTITUTIONS",
        "WB_2290_OIL_AND_GAS_EXPORT",
        "WB_544_MID_AND_DOWNSTREAM_OIL_AND_GAS",
        "WB_2273_UPSTREAM_OIL_AND_GAS",
        "WB_525_RENEWABLE_ENERGY",
        "WB_532_BIOFUELS_ENERGY",
        "WB_548_PPP_IN_OIL_AND_GAS"
    },

    "Sanctions & Economic Security": {
        "SANCTIONS"
    },

    "Conflict & Geopolitical Tensions": {
        "ARMEDCONFLICT",
        "WB_2433_CONFLICT_AND_VIOLENCE",
        "WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE",
        "WB_739_POLITICAL_VIOLENCE_AND_CIVIL_WAR",
        "WB_2462_POLITICAL_VIOLENCE_AND_WAR",
        "TERROR"
    }
}

In [102]:
def classify_gkg_security_domains(theme_list):
    """
    Classifies an article into one or more security domains
    using high-confidence GDELT GKG themes.
    """

    if not isinstance(theme_list, list):
        return []

    detected_domains = []

    theme_set = set(theme_list)

    for domain, relevant_themes in GKG_SECURITY_THEMES.items():

        if theme_set.intersection(relevant_themes):
            detected_domains.append(domain)

    return detected_domains

In [103]:
df_security["theme_list"] = (
    df_security["themes"]
    .apply(split_gkg_themes)
)

In [104]:
df_security["security_domains"] = (
    df_security["theme_list"]
    .apply(classify_gkg_security_domains)
)

In [105]:
df_security["security_relevant"] = (
    df_security["security_domains"]
    .apply(lambda x: len(x) > 0)
)

In [106]:
df_relevant = df_security[
    df_security["security_relevant"]
].copy()

df_not_relevant = df_security[
    ~df_security["security_relevant"]
].copy()

In [107]:
print("European / strategic events:", len(df_security))

print(
    "Security relevant events:",
    len(df_relevant)
)

print(
    "Excluded events:",
    len(df_not_relevant)
)

print(
    "Security relevance rate:",
    round(
        len(df_relevant)
        / len(df_security)
        * 100,
        2
    ),
    "%"
)

European / strategic events: 267
Security relevant events: 180
Excluded events: 87
Security relevance rate: 67.42 %


In [108]:
domain_distribution = (
    df_relevant["security_domains"]
    .explode()
    .value_counts()
    .reset_index()
)

domain_distribution.columns = [
    "security_domain",
    "events"
]

domain_distribution

,security_domain,events
0,Conflict & Geopolitical Tensions,130
1,Defence & Military,117
2,Energy Security,69
3,Cybersecurity,13
4,Sanctions & Economic Security,6


In [109]:
relevance_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor2",
            "event_root_label",
            "location",
            "attention_score",
            "security_domains",
            "security_relevant",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

relevance_check

,event_date,actor1,actor2,event_root_label,location,attention_score,security_domains,security_relevant,source_url
21,2026-08-24,NaN,ITALY,Fight,Ukraine,94.30,"[Defence & Military, Cybersecurity, Conflict & Geopolitical Tensions]",True,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/
43,2026-08-24,BRAZILIAN,LIMERICK,Fight,"Sorocaba, SãPaulo, Brazil",87.90,[],False,https://www.limerickpost.ie/2026/08/24/gardai-treating-death-of-brazilian-woman-in-limerick-as-murder-suicide-after-suspects-body-found/
117,2026-08-24,UNITED KINGDOM,NaN,Assault,"Hemel Hempstead, Hertfordshire, United Kingdom",86.52,[],False,https://www.hertsad.co.uk/news/26489327.herts-teacher-accused-sex-pupils-was-just-friendly/
120,2026-08-24,GLASGOW,NaN,Fight,"Glasgow, Glasgow City, United Kingdom",86.42,[],False,https://www.dailyrecord.co.uk/news/scottish-news/glasgow-murder-cops-seize-4000-37586658
258,2026-08-24,UKRAINIAN,RUSSIAN,Fight,"Kyiv, Kyyiv, Misto, Ukraine",85.38,"[Defence & Military, Energy Security, Conflict & Geopolitical Tensions]",True,https://www.kyivpost.com/opinion/82914
119,2026-08-24,EDINBURGH,NaN,Fight,"Edinburgh, Edinburgh, City of, United Kingdom",85.23,[],False,https://www.dailyrecord.co.uk/tv/mesmerising-bbc-thriller-set-rural-37585853
234,2026-08-24,RUSSIAN,UKRAINE,Fight,"Vede, Russia (general), Russia",84.92,"[Defence & Military, Conflict & Geopolitical Tensions]",True,https://www.kyivpost.com/post/82985
19,2026-08-24,NaN,UNITED KINGDOM,Fight,"Cambridge, Cambridgeshire, United Kingdom",82.40,[],False,https://www.christiantoday.com/news/learning-from-the-jason-arday-tragedy
136,2026-08-24,BRITAIN,RUSSIA,Fight,"Kremlin, Moskva, Russia",80.09,[Energy Security],True,https://www.bangkokpost.com/world/3307108/kremlin-says-uk-preventing-peace-by-giving-missile-info-to-ukraine
153,2026-08-24,FIRE COMPANY,NaN,Fight,"Smyrna, Izmir, Turkey",78.29,[],False,https://www.eaglecountryonline.com/news/local-news/napoleon-fire-mourning-the-loss-of-assistant-fire-chief/


## 18. Event Actor Context

In [110]:
actor_context = df_gdelt[
    [
        "GLOBALEVENTID",
        "Actor1Type1Code",
        "Actor2Type1Code",
        "Actor1KnownGroupCode",
        "Actor2KnownGroupCode"
    ]
].copy()

actor_context = actor_context.rename(columns={
    "GLOBALEVENTID": "event_id",
    "Actor1Type1Code": "actor1_type",
    "Actor2Type1Code": "actor2_type",
    "Actor1KnownGroupCode": "actor1_group",
    "Actor2KnownGroupCode": "actor2_group"
})

In [111]:
print(
    "Rows:",
    len(actor_context)
)

print(
    "Unique event IDs:",
    actor_context["event_id"].nunique()
)

Rows: 1175
Unique event IDs: 1175


In [112]:
df_security = df_security.merge(
    actor_context,
    on="event_id",
    how="left",
    validate="one_to_one"
)

In [113]:
print("Rows after actor merge:", len(df_security))

Rows after actor merge: 267


In [114]:
actor_relevance_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor1_country",
            "actor1_type",
            "actor1_group",
            "actor2",
            "actor2_country",
            "actor2_type",
            "actor2_group",
            "event_root_label",
            "location",
            "attention_score",
            "security_domains",
            "security_relevant",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

actor_relevance_check

,event_date,actor1,actor1_country,actor1_type,actor1_group,actor2,actor2_country,actor2_type,actor2_group,event_root_label,location,attention_score,security_domains,security_relevant,source_url
21,2026-08-24,NaN,NaN,NaN,NaN,ITALY,ITA,NaN,NaN,Fight,Ukraine,94.30,"[Defence & Military, Cybersecurity, Conflict & Geopolitical Tensions]",True,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/
43,2026-08-24,BRAZILIAN,BRA,NaN,NaN,LIMERICK,IRL,NaN,NaN,Fight,"Sorocaba, SãPaulo, Brazil",87.90,[],False,https://www.limerickpost.ie/2026/08/24/gardai-treating-death-of-brazilian-woman-in-limerick-as-murder-suicide-after-suspects-body-found/
117,2026-08-24,UNITED KINGDOM,GBR,NaN,NaN,NaN,NaN,NaN,NaN,Assault,"Hemel Hempstead, Hertfordshire, United Kingdom",86.52,[],False,https://www.hertsad.co.uk/news/26489327.herts-teacher-accused-sex-pupils-was-just-friendly/
120,2026-08-24,GLASGOW,GBR,NaN,NaN,NaN,NaN,NaN,NaN,Fight,"Glasgow, Glasgow City, United Kingdom",86.42,[],False,https://www.dailyrecord.co.uk/news/scottish-news/glasgow-murder-cops-seize-4000-37586658
258,2026-08-24,UKRAINIAN,UKR,UAF,NaN,RUSSIAN,RUS,NaN,NaN,Fight,"Kyiv, Kyyiv, Misto, Ukraine",85.38,"[Defence & Military, Energy Security, Conflict & Geopolitical Tensions]",True,https://www.kyivpost.com/opinion/82914
119,2026-08-24,EDINBURGH,GBR,NaN,NaN,NaN,NaN,NaN,NaN,Fight,"Edinburgh, Edinburgh, City of, United Kingdom",85.23,[],False,https://www.dailyrecord.co.uk/tv/mesmerising-bbc-thriller-set-rural-37585853
234,2026-08-24,RUSSIAN,RUS,NaN,NaN,UKRAINE,UKR,NaN,NaN,Fight,"Vede, Russia (general), Russia",84.92,"[Defence & Military, Conflict & Geopolitical Tensions]",True,https://www.kyivpost.com/post/82985
19,2026-08-24,NaN,NaN,NaN,NaN,UNITED KINGDOM,GBR,NaN,NaN,Fight,"Cambridge, Cambridgeshire, United Kingdom",82.40,[],False,https://www.christiantoday.com/news/learning-from-the-jason-arday-tragedy
136,2026-08-24,BRITAIN,GBR,NaN,NaN,RUSSIA,RUS,NaN,NaN,Fight,"Kremlin, Moskva, Russia",80.09,[Energy Security],True,https://www.bangkokpost.com/world/3307108/kremlin-says-uk-preventing-peace-by-giving-missile-info-to-ukraine
153,2026-08-24,FIRE COMPANY,NaN,GOV,NaN,NaN,NaN,NaN,NaN,Fight,"Smyrna, Izmir, Turkey",78.29,[],False,https://www.eaglecountryonline.com/news/local-news/napoleon-fire-mourning-the-loss-of-assistant-fire-chief/


In [115]:
print("Actor 1 types:")

print(
    df_security["actor1_type"]
    .value_counts(dropna=False)
    .head(30)
)

Actor 1 types:
actor1_type
NaN    177
GOV     39
EDU     12
IGO     10
MED      8
CVL      7
MIL      4
MNC      3
BUS      2
INT      2
COP      1
CRM      1
UAF      1
Name: count, dtype: int64


In [116]:
print("Actor 2 types:")

print(
    df_security["actor2_type"]
    .value_counts(dropna=False)
    .head(30)
)

Actor 2 types:
actor2_type
NaN    204
GOV     26
EDU     12
IGO      7
BUS      7
CVL      6
ELI      1
LAB      1
COP      1
MNC      1
MIL      1
Name: count, dtype: int64


In [117]:
STRATEGIC_ACTOR_TYPES = {
    "GOV",  # Government
    "MIL",  # Military
    "REB",  # Rebels
    "INS",  # Insurgents
    "SEP",  # Separatists
    "SPY",  # Intelligence services
    "UAF",  # Unaligned armed forces
    "IGO"   # International governmental organisations
}

In [118]:
df_security["strategic_actor"] = (
    df_security["actor1_type"].isin(STRATEGIC_ACTOR_TYPES)
    |
    df_security["actor2_type"].isin(STRATEGIC_ACTOR_TYPES)
)

In [119]:
print(
    "Events with strategic actor:",
    df_security["strategic_actor"].sum()
)

Events with strategic actor: 78


In [120]:
COUNTRY_TO_CAMEO = {
    "Albania": "ALB",
    "Austria": "AUT",
    "Belarus": "BLR",
    "Belgium": "BEL",
    "Bosnia and Herzegovina": "BIH",
    "Bulgaria": "BGR",
    "Croatia": "HRV",
    "Cyprus": "CYP",
    "Czechia": "CZE",
    "Denmark": "DNK",
    "Estonia": "EST",
    "Finland": "FIN",
    "France": "FRA",
    "Germany": "DEU",
    "Greece": "GRC",
    "Hungary": "HUN",
    "Iceland": "ISL",
    "Ireland": "IRL",
    "Italy": "ITA",
    "Latvia": "LVA",
    "Lithuania": "LTU",
    "Luxembourg": "LUX",
    "Malta": "MLT",
    "Moldova": "MDA",
    "Montenegro": "MNE",
    "Netherlands": "NLD",
    "North Macedonia": "MKD",
    "Norway": "NOR",
    "Poland": "POL",
    "Portugal": "PRT",
    "Romania": "ROU",
    "Serbia": "SRB",
    "Slovakia": "SVK",
    "Slovenia": "SVN",
    "Spain": "ESP",
    "Sweden": "SWE",
    "Switzerland": "CHE",
    "Türkiye": "TUR",
    "Ukraine": "UKR",
    "United Kingdom": "GBR",
    "Russia": "RUS",
    "Georgia": "GEO",
    "Armenia": "ARM",
    "Azerbaijan": "AZE",
    "Monaco": "MCO",
    "Andorra": "AND",
    "Liechtenstein": "LIE",
    "San Marino": "SMR",
    "Vatican City": "VAT"
}

In [121]:
def get_location_cameo(location_countries):
    
    if not isinstance(location_countries, list):
        return None
    
    if len(location_countries) == 0:
        return None
    
    country = location_countries[0]
    
    return COUNTRY_TO_CAMEO.get(country)

In [122]:
df_security["location_cameo"] = (
    df_security["location_countries"]
    .apply(get_location_cameo)
)

In [123]:
def detect_cross_border_context(row):
    
    location_code = row["location_cameo"]
    
    if pd.isna(location_code):
        return False
    
    actor_codes = [
        row["actor1_country"],
        row["actor2_country"]
    ]
    
    actor_codes = [
        code
        for code in actor_codes
        if pd.notna(code)
    ]
    
    for actor_code in actor_codes:
        
        if (
            actor_code in MONITORED_ISO3
            and actor_code != location_code
        ):
            return True
    
    return False

In [124]:
df_security["cross_border_context"] = (
    df_security.apply(
        detect_cross_border_context,
        axis=1
    )
)

In [125]:
df_security["strategic_context"] = (
    df_security["strategic_actor"]
    |
    df_security["cross_border_context"]
)

In [126]:
print(
    "Strategic actor:",
    df_security["strategic_actor"].sum()
)

print(
    "Cross-border context:",
    df_security["cross_border_context"].sum()
)

print(
    "Strategic context total:",
    df_security["strategic_context"].sum()
)

Strategic actor: 78
Cross-border context: 52
Strategic context total: 117


In [127]:
DIRECT_DEFENCE_THEMES = {
    "MILITARY_COOPERATION",
    "TAX_WEAPONS_DRONE_STRIKE",
    "TAX_WEAPONS_ARTILLERY"
}

BROAD_DEFENCE_THEMES = {
    "MILITARY",
    "ARMEDCONFLICT",
    "WB_2470_PEACE_OPERATIONS_AND_CONFLICT_MANAGEMENT"
}


CYBER_THEMES = {
    "CYBER_ATTACK",
    "WB_670_ICT_SECURITY",
    "TAX_FNCACT_HACKER",
    "TAX_FNCACT_HACKERS"
}


SANCTIONS_THEMES = {
    "SANCTIONS"
}


DIRECT_CONFLICT_THEMES = {
    "WB_739_POLITICAL_VIOLENCE_AND_CIVIL_WAR",
    "WB_2462_POLITICAL_VIOLENCE_AND_WAR",
    "TERROR"
}

BROAD_CONFLICT_THEMES = {
    "ARMEDCONFLICT",
    "WB_2433_CONFLICT_AND_VIOLENCE",
    "WB_2432_FRAGILITY_CONFLICT_AND_VIOLENCE"
}


ENERGY_THEMES = {
    "WB_507_ENERGY_AND_EXTRACTIVES",
    "ENV_OIL",
    "ENV_NATURALGAS",
    "WB_539_OIL_AND_GAS_POLICY_STRATEGY_AND_INSTITUTIONS",
    "WB_2290_OIL_AND_GAS_EXPORT",
    "WB_544_MID_AND_DOWNSTREAM_OIL_AND_GAS",
    "WB_2273_UPSTREAM_OIL_AND_GAS",
    "WB_525_RENEWABLE_ENERGY",
    "WB_532_BIOFUELS_ENERGY",
    "WB_548_PPP_IN_OIL_AND_GAS"
}

SECURITY_CONFLICT_ROOTS = {
    13,  # Threaten
    15,  # Exhibit Force Posture
    16,  # Reduce Relations
    17,  # Coerce
    18,  # Assault
    19,  # Fight
    20   # Unconventional Mass Violence
}

In [128]:
def classify_security_domains_v2(row):
    
    themes = set(row["theme_list"])
    
    domains = []
    
    strategic_context = row["strategic_context"]
    root_code = row["event_root_code"]
    
    
    # -------------------------
    # CYBERSECURITY
    # -------------------------
    
    if themes.intersection(CYBER_THEMES):
        domains.append("Cybersecurity")
    
    
    # -------------------------
    # SANCTIONS
    # -------------------------
    
    if themes.intersection(SANCTIONS_THEMES):
        domains.append(
            "Sanctions & Economic Security"
        )
    
    
    # -------------------------
    # DEFENCE & MILITARY
    # -------------------------
    
    direct_defence = bool(
        themes.intersection(
            DIRECT_DEFENCE_THEMES
        )
    )
    
    contextual_defence = (
        bool(
            themes.intersection(
                BROAD_DEFENCE_THEMES
            )
        )
        and strategic_context
    )
    
    if direct_defence or contextual_defence:
        domains.append(
            "Defence & Military"
        )
    
    
    # -------------------------
    # CONFLICT
    # -------------------------
    
    direct_conflict = bool(
        themes.intersection(
            DIRECT_CONFLICT_THEMES
        )
    )
    
    contextual_conflict = (
        bool(
            themes.intersection(
                BROAD_CONFLICT_THEMES
            )
        )
        and strategic_context
        and root_code in SECURITY_CONFLICT_ROOTS
    )
    
    if direct_conflict or contextual_conflict:
        domains.append(
            "Conflict & Geopolitical Tensions"
        )
    
    
    # -------------------------
    # ENERGY SECURITY
    # -------------------------
    
    energy_theme = bool(
        themes.intersection(
            ENERGY_THEMES
        )
    )
    
    security_context = (
        strategic_context
        or bool(
            themes.intersection(
                SANCTIONS_THEMES
                | BROAD_CONFLICT_THEMES
                | DIRECT_CONFLICT_THEMES
            )
        )
    )
    
    if energy_theme and security_context:
        domains.append(
            "Energy Security"
        )
    
    
    return domains

In [129]:
df_security["security_domains_v2"] = (
    df_security.apply(
        classify_security_domains_v2,
        axis=1
    )
)

df_security["security_relevant_v2"] = (
    df_security["security_domains_v2"]
    .apply(lambda x: len(x) > 0)
)

In [130]:
print(
    "V1 relevant:",
    df_security["security_relevant"].sum()
)

print(
    "V2 relevant:",
    df_security["security_relevant_v2"].sum()
)

V1 relevant: 180
V2 relevant: 114


In [131]:
relevance_v2_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor1_country",
            "actor1_type",
            "actor2",
            "actor2_country",
            "actor2_type",
            "event_root_label",
            "location",
            "strategic_actor",
            "cross_border_context",
            "attention_score",
            "security_domains_v2",
            "security_relevant_v2",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

relevance_v2_check

,event_date,actor1,actor1_country,actor1_type,actor2,actor2_country,actor2_type,event_root_label,location,strategic_actor,cross_border_context,attention_score,security_domains_v2,security_relevant_v2,source_url
21,2026-08-24,NaN,NaN,NaN,ITALY,ITA,NaN,Fight,Ukraine,False,True,94.30,"[Cybersecurity, Defence & Military, Conflict & Geopolitical Tensions]",True,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/
43,2026-08-24,BRAZILIAN,BRA,NaN,LIMERICK,IRL,NaN,Fight,"Sorocaba, SãPaulo, Brazil",False,False,87.90,[],False,https://www.limerickpost.ie/2026/08/24/gardai-treating-death-of-brazilian-woman-in-limerick-as-murder-suicide-after-suspects-body-found/
117,2026-08-24,UNITED KINGDOM,GBR,NaN,NaN,NaN,NaN,Assault,"Hemel Hempstead, Hertfordshire, United Kingdom",False,False,86.52,[],False,https://www.hertsad.co.uk/news/26489327.herts-teacher-accused-sex-pupils-was-just-friendly/
120,2026-08-24,GLASGOW,GBR,NaN,NaN,NaN,NaN,Fight,"Glasgow, Glasgow City, United Kingdom",False,False,86.42,[],False,https://www.dailyrecord.co.uk/news/scottish-news/glasgow-murder-cops-seize-4000-37586658
258,2026-08-24,UKRAINIAN,UKR,UAF,RUSSIAN,RUS,NaN,Fight,"Kyiv, Kyyiv, Misto, Ukraine",True,True,85.38,"[Defence & Military, Conflict & Geopolitical Tensions, Energy Security]",True,https://www.kyivpost.com/opinion/82914
119,2026-08-24,EDINBURGH,GBR,NaN,NaN,NaN,NaN,Fight,"Edinburgh, Edinburgh, City of, United Kingdom",False,False,85.23,[],False,https://www.dailyrecord.co.uk/tv/mesmerising-bbc-thriller-set-rural-37585853
234,2026-08-24,RUSSIAN,RUS,NaN,UKRAINE,UKR,NaN,Fight,"Vede, Russia (general), Russia",False,True,84.92,"[Defence & Military, Conflict & Geopolitical Tensions]",True,https://www.kyivpost.com/post/82985
19,2026-08-24,NaN,NaN,NaN,UNITED KINGDOM,GBR,NaN,Fight,"Cambridge, Cambridgeshire, United Kingdom",False,False,82.40,[],False,https://www.christiantoday.com/news/learning-from-the-jason-arday-tragedy
136,2026-08-24,BRITAIN,GBR,NaN,RUSSIA,RUS,NaN,Fight,"Kremlin, Moskva, Russia",False,True,80.09,[Energy Security],True,https://www.bangkokpost.com/world/3307108/kremlin-says-uk-preventing-peace-by-giving-missile-info-to-ukraine
153,2026-08-24,FIRE COMPANY,NaN,GOV,NaN,NaN,NaN,Fight,"Smyrna, Izmir, Turkey",True,False,78.29,[],False,https://www.eaglecountryonline.com/news/local-news/napoleon-fire-mourning-the-loss-of-assistant-fire-chief/


In [132]:
from urllib.parse import urlparse, unquote
import re


def url_to_text(url):
    """
    Converts a news URL path into a text proxy
    that can be used for high-precision keyword matching.
    """

    if not isinstance(url, str):
        return ""

    path = unquote(urlparse(url).path)

    text = re.sub(
        r"[-_/]+",
        " ",
        path
    )

    text = re.sub(
        r"\b\d+\b",
        " ",
        text
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip().lower()

In [133]:
df_security["url_text"] = (
    df_security["source_url"]
    .apply(url_to_text)
)

In [134]:
df_security[
    [
        "source_url",
        "url_text",
        "attention_score"
    ]
].sort_values(
    "attention_score",
    ascending=False
).head(10)

,source_url,url_text,attention_score
21,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/,russias shadow war moves inside europes arms factories,94.30
66,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/,russias shadow war moves inside europes arms factories,94.30
43,https://www.limerickpost.ie/2026/08/24/gardai-treating-death-of-brazilian-woman-in-limerick-as-murder-suicide-after-suspects-body-found/,gardai treating death of brazilian woman in limerick as murder suicide after suspects body found,87.90
117,https://www.hertsad.co.uk/news/26489327.herts-teacher-accused-sex-pupils-was-just-friendly/,news .herts teacher accused sex pupils was just friendly,86.52
120,https://www.dailyrecord.co.uk/news/scottish-news/glasgow-murder-cops-seize-4000-37586658,news scottish news glasgow murder cops seize,86.42
258,https://www.kyivpost.com/opinion/82914,opinion,85.38
119,https://www.dailyrecord.co.uk/tv/mesmerising-bbc-thriller-set-rural-37585853,tv mesmerising bbc thriller set rural,85.23
234,https://www.kyivpost.com/post/82985,post,84.92
196,https://www.kyivpost.com/post/82985,post,84.37
39,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/,russias shadow war moves inside europes arms factories,83.46


In [135]:
URL_SECURITY_KEYWORDS = {

    "Defence & Military": [
        "military",
        "armed forces",
        "air defence",
        "air defense",
        "missile",
        "missile strike",
        "drone strike",
        "drone strikes",
        "artillery",
        "troops",
        "nato",
        "defence ministry",
        "defense ministry"
    ],

    "Cybersecurity": [
        "cyberattack",
        "cyber attack",
        "cybersecurity",
        "ransomware",
        "malware",
        "hacking",
        "data breach"
    ],

    "Energy Security": [
        "energy security",
        "gas pipeline",
        "oil pipeline",
        "natural gas",
        "energy infrastructure",
        "power grid",
        "oil and gas"
    ],

    "Sanctions & Economic Security": [
        "sanctions",
        "economic sanctions",
        "export controls",
        "asset freeze",
        "embargo"
    ],

    "Conflict & Geopolitical Tensions": [
        "armed conflict",
        "military escalation",
        "invasion",
        "ceasefire",
        "hostilities",
        "airstrike",
        "air strike",
        "missile strike",
        "drone strike",
        "drone strikes",
        "shelling",
        "frontline",
        "front line"
    ]
}

In [136]:
def classify_url_security(text):

    if not isinstance(text, str):
        return []

    detected = []

    for domain, keywords in URL_SECURITY_KEYWORDS.items():

        for keyword in keywords:

            if keyword in text:
                detected.append(domain)
                break

    return detected

In [137]:
df_security["url_security_domains"] = (
    df_security["url_text"]
    .apply(classify_url_security)
)

In [138]:
df_security["url_countries"] = (
    df_security["url_text"]
    .apply(extract_countries)
)

In [139]:
def black_sea_security_context(row):

    text = row["url_text"]
    countries = row["url_countries"]

    if "black sea" not in text:
        return False

    strategic_black_sea_countries = {
        "Ukraine",
        "Russia",
        "Türkiye"
    }

    return bool(
        strategic_black_sea_countries
        .intersection(set(countries))
    )

In [140]:
df_security["black_sea_context"] = (
    df_security.apply(
        black_sea_security_context,
        axis=1
    )
)

In [141]:
def classify_security_domains_v3(row):

    themes = set(row["theme_list"])

    url_domains = set(
        row["url_security_domains"]
    )

    strategic_context = row["strategic_context"]

    black_sea_context = row["black_sea_context"]

    root_code = row["event_root_code"]

    domains = []


    # ----------------------------------
    # CYBERSECURITY
    # ----------------------------------

    if (
        themes.intersection(CYBER_THEMES)
        or "Cybersecurity" in url_domains
    ):
        domains.append(
            "Cybersecurity"
        )


    # ----------------------------------
    # SANCTIONS
    # ----------------------------------

    if (
        themes.intersection(SANCTIONS_THEMES)
        or "Sanctions & Economic Security"
        in url_domains
    ):
        domains.append(
            "Sanctions & Economic Security"
        )


    # ----------------------------------
    # DEFENCE & MILITARY
    # ----------------------------------

    defence_theme = bool(
        themes.intersection(
            DIRECT_DEFENCE_THEMES
            | BROAD_DEFENCE_THEMES
        )
    )

    defence_url = (
        "Defence & Military"
        in url_domains
    )

    if (
        defence_url
        or (
            defence_theme
            and (
                strategic_context
                or black_sea_context
            )
        )
    ):
        domains.append(
            "Defence & Military"
        )


    # ----------------------------------
    # CONFLICT / GEOPOLITICAL
    # ----------------------------------

    conflict_theme = bool(
        themes.intersection(
            DIRECT_CONFLICT_THEMES
            | BROAD_CONFLICT_THEMES
        )
    )

    conflict_url = (
        "Conflict & Geopolitical Tensions"
        in url_domains
    )

    conflict_event = (
        root_code
        in SECURITY_CONFLICT_ROOTS
    )

    if (
        conflict_url
        or (
            conflict_theme
            and conflict_event
            and (
                strategic_context
                or black_sea_context
            )
        )
    ):
        domains.append(
            "Conflict & Geopolitical Tensions"
        )


    # ----------------------------------
    # ENERGY SECURITY
    # ----------------------------------

    energy_theme = bool(
        themes.intersection(
            ENERGY_THEMES
        )
    )

    energy_url = (
        "Energy Security"
        in url_domains
    )

    if (
        energy_url
        or (
            energy_theme
            and (
                strategic_context
                or black_sea_context
                or bool(
                    themes.intersection(
                        SANCTIONS_THEMES
                        | BROAD_CONFLICT_THEMES
                        | DIRECT_CONFLICT_THEMES
                    )
                )
            )
        )
    ):
        domains.append(
            "Energy Security"
        )


    return list(dict.fromkeys(domains))

In [142]:
df_security["security_domains_v3"] = (
    df_security.apply(
        classify_security_domains_v3,
        axis=1
    )
)

df_security["security_relevant_v3"] = (
    df_security["security_domains_v3"]
    .apply(lambda x: len(x) > 0)
)

In [143]:
print(
    "V1 relevant:",
    df_security["security_relevant"].sum()
)

print(
    "V2 relevant:",
    df_security["security_relevant_v2"].sum()
)

print(
    "V3 relevant:",
    df_security["security_relevant_v3"].sum()
)

V1 relevant: 180
V2 relevant: 114
V3 relevant: 114


In [144]:
relevance_v3_check = (
    df_security[
        [
            "event_date",
            "actor1",
            "actor1_country",
            "actor1_type",
            "actor2",
            "actor2_country",
            "actor2_type",
            "event_root_label",
            "location",
            "attention_score",
            "strategic_context",
            "url_text",
            "url_security_domains",
            "security_domains_v3",
            "security_relevant_v3",
            "source_url"
        ]
    ]
    .sort_values(
        "attention_score",
        ascending=False
    )
    .drop_duplicates(
        subset="source_url"
    )
    .head(20)
)

relevance_v3_check

,event_date,actor1,actor1_country,actor1_type,actor2,actor2_country,actor2_type,event_root_label,location,attention_score,strategic_context,url_text,url_security_domains,security_domains_v3,security_relevant_v3,source_url
21,2026-08-24,NaN,NaN,NaN,ITALY,ITA,NaN,Fight,Ukraine,94.30,True,russias shadow war moves inside europes arms factories,[],"[Cybersecurity, Defence & Military, Conflict & Geopolitical Tensions]",True,https://londonlovesbusiness.com/russias-shadow-war-moves-inside-europes-arms-factories/
43,2026-08-24,BRAZILIAN,BRA,NaN,LIMERICK,IRL,NaN,Fight,"Sorocaba, SãPaulo, Brazil",87.90,False,gardai treating death of brazilian woman in limerick as murder suicide after suspects body found,[],[],False,https://www.limerickpost.ie/2026/08/24/gardai-treating-death-of-brazilian-woman-in-limerick-as-murder-suicide-after-suspects-body-found/
117,2026-08-24,UNITED KINGDOM,GBR,NaN,NaN,NaN,NaN,Assault,"Hemel Hempstead, Hertfordshire, United Kingdom",86.52,False,news .herts teacher accused sex pupils was just friendly,[],[],False,https://www.hertsad.co.uk/news/26489327.herts-teacher-accused-sex-pupils-was-just-friendly/
120,2026-08-24,GLASGOW,GBR,NaN,NaN,NaN,NaN,Fight,"Glasgow, Glasgow City, United Kingdom",86.42,False,news scottish news glasgow murder cops seize,[],[],False,https://www.dailyrecord.co.uk/news/scottish-news/glasgow-murder-cops-seize-4000-37586658
258,2026-08-24,UKRAINIAN,UKR,UAF,RUSSIAN,RUS,NaN,Fight,"Kyiv, Kyyiv, Misto, Ukraine",85.38,True,opinion,[],"[Defence & Military, Conflict & Geopolitical Tensions, Energy Security]",True,https://www.kyivpost.com/opinion/82914
119,2026-08-24,EDINBURGH,GBR,NaN,NaN,NaN,NaN,Fight,"Edinburgh, Edinburgh, City of, United Kingdom",85.23,False,tv mesmerising bbc thriller set rural,[],[],False,https://www.dailyrecord.co.uk/tv/mesmerising-bbc-thriller-set-rural-37585853
234,2026-08-24,RUSSIAN,RUS,NaN,UKRAINE,UKR,NaN,Fight,"Vede, Russia (general), Russia",84.92,True,post,[],"[Defence & Military, Conflict & Geopolitical Tensions]",True,https://www.kyivpost.com/post/82985
19,2026-08-24,NaN,NaN,NaN,UNITED KINGDOM,GBR,NaN,Fight,"Cambridge, Cambridgeshire, United Kingdom",82.40,False,news learning from the jason arday tragedy,[],[],False,https://www.christiantoday.com/news/learning-from-the-jason-arday-tragedy
136,2026-08-24,BRITAIN,GBR,NaN,RUSSIA,RUS,NaN,Fight,"Kremlin, Moskva, Russia",80.09,True,world kremlin says uk preventing peace by giving missile info to ukraine,[Defence & Military],"[Defence & Military, Energy Security]",True,https://www.bangkokpost.com/world/3307108/kremlin-says-uk-preventing-peace-by-giving-missile-info-to-ukraine
153,2026-08-24,FIRE COMPANY,NaN,GOV,NaN,NaN,NaN,Fight,"Smyrna, Izmir, Turkey",78.29,True,news local news napoleon fire mourning the loss of assistant fire chief,[],[],False,https://www.eaglecountryonline.com/news/local-news/napoleon-fire-mourning-the-loss-of-assistant-fire-chief/


## 21. Final Security-Relevant Dataset

In [145]:
df_relevant = df_security[
    df_security["security_relevant_v3"]
].copy()

print("Total European / strategic events:", len(df_security))
print("Final security-relevant events:", len(df_relevant))

Total European / strategic events: 267
Final security-relevant events: 114


In [146]:
domain_distribution_v3 = (
    df_relevant["security_domains_v3"]
    .explode()
    .value_counts()
    .reset_index()
)

domain_distribution_v3.columns = [
    "security_domain",
    "events"
]

domain_distribution_v3

,security_domain,events
0,Defence & Military,93
1,Energy Security,33
2,Conflict & Geopolitical Tensions,24
3,Cybersecurity,13
4,Sanctions & Economic Security,9


## 22. SQLite Database

In [147]:
import sqlite3

In [148]:
database_path = (
    project_root
    / "data"
    / "security_monitor.db"
)

print(database_path)

c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\security_monitor.db


In [149]:
conn = sqlite3.connect(database_path)

print("SQLite connection created")

SQLite connection created


In [150]:
["Defence & Military", "Conflict & Geopolitical Tensions"]

['Defence & Military', 'Conflict & Geopolitical Tensions']

In [151]:
df_sql = df_relevant.copy()

In [152]:
df_sql["security_domains"] = (
    df_sql["security_domains_v3"]
    .apply(lambda x: " | ".join(x))
)

In [153]:
df_sql[
    [
        "security_domains_v3",
        "security_domains"
    ]
].head()

,security_domains_v3,security_domains
8,[Defence & Military],Defence & Military
9,[Defence & Military],Defence & Military
10,[Defence & Military],Defence & Military
11,[Defence & Military],Defence & Military
18,[Defence & Military],Defence & Military


In [154]:
security_event_columns = [
    "event_id",
    "event_date",

    "actor1",
    "actor1_country",
    "actor1_type",

    "actor2",
    "actor2_country",
    "actor2_type",

    "event_code",
    "event_root_code",
    "event_root_label",
    "quad_class",
    "quad_class_label",

    "goldstein_scale",
    "avg_tone",

    "num_mentions",
    "num_articles",

    "location",
    "latitude",
    "longitude",

    "security_domains",

    "attention_score",
    "attention_band",

    "source_name",
    "source_url"
]

In [155]:
df_sql_events = df_sql[
    security_event_columns
].copy()

In [156]:
print(df_sql_events.shape)

df_sql_events.head()

(114, 25)


,event_id,event_date,actor1,actor1_country,actor1_type,actor2,actor2_country,actor2_type,event_code,event_root_code,...,num_mentions,num_articles,location,latitude,longitude,security_domains,attention_score,attention_band,source_name,source_url
8,1319682905,2026-08-17,GOVERNMENT SPOKESMAN,NaN,GOV,FRANCE,FRA,NaN,10,1,...,2,2,"Paris, France (general), France",48.8667,2.33333,Defence & Military,31.47,Low,egyptindependent.com,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
9,1319682906,2026-08-17,GOVERNMENT SPOKESMAN,NaN,GOV,FRENCH,FRA,NaN,10,1,...,2,2,"Paris, France (general), France",48.8667,2.33333,Defence & Military,31.47,Low,egyptindependent.com,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
10,1319682907,2026-08-17,GOVERNMENT SPOKESMAN,NaN,GOV,FRENCH,FRA,NaN,10,1,...,1,1,"Maroua, Extreme-Nord, Cameroon",10.5909,14.31590,Defence & Military,28.09,Low,egyptindependent.com,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
11,1319682912,2026-08-17,POLAND,POL,NaN,AMBASSADOR,NaN,GOV,40,4,...,1,1,"Warsaw, (PL67), Poland",52.2500,21.00000,Defence & Military,30.02,Low,forward.com,https://forward.com/fast-forward/846473/israel-slammed-for-closing-probe-into-killing-of-seven-world-central-kitchen-workers/
18,1319682962,2026-08-24,NaN,NaN,NaN,LONDON,GBR,NaN,112,11,...,2,2,"Kyiv, Kyyiv, Misto, Ukraine",50.4333,30.51670,Defence & Military,49.47,Medium,londonlovesbusiness.com,https://londonlovesbusiness.com/kremlin-warns-britain-over-burnhams-secret-storm-shadow-plans/


In [157]:
print("Rows:", len(df_sql_events))
print("Columns:", len(df_sql_events.columns))
print("Database:", database_path)

Rows: 114
Columns: 25
Database: c:\Users\cl_am\OneDrive\Desktop\european-security-monitor\data\security_monitor.db


## 23. Store Security Events in SQLite

In [158]:
if_exists="replace"

In [159]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

tables

,name
0,security_events


In [160]:
df_sql_events.to_sql(
    "security_events",
    conn,
    if_exists="replace",
    index=False
)

print("Table 'security_events' created successfully")

Table 'security_events' created successfully


In [161]:
tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table';
    """,
    conn
)

tables

,name
0,security_events


In [162]:
row_count = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_events
    FROM security_events;
    """,
    conn
)

row_count

,total_events
0,114


## 24. SQL Analysis

In [163]:
query = """
SELECT
    attention_band,
    COUNT(*) AS events
FROM security_events
GROUP BY attention_band
ORDER BY events DESC;
"""

attention_sql = pd.read_sql_query(
    query,
    conn
)

attention_sql

,attention_band,events
0,Low,50
1,Medium,30
2,High,20
3,Critical,14


In [164]:
query = """
SELECT
    event_date,
    actor1,
    actor2,
    event_root_label,
    location,
    security_domains,
    attention_score,
    attention_band
FROM security_events
ORDER BY attention_score DESC
LIMIT 10;
"""

top_events_sql = pd.read_sql_query(
    query,
    conn
)

top_events_sql

,event_date,actor1,actor2,event_root_label,location,security_domains,attention_score,attention_band
0,2026-08-24 00:00:00,NaN,ITALY,Fight,Ukraine,Cybersecurity | Defence & Military | Conflict & Geopolitical Tensions,94.30,Critical
1,2026-08-24 00:00:00,CRIMINAL,NaN,Fight,"Moscow, Moskva, Russia",Cybersecurity,94.30,Critical
2,2026-08-24 00:00:00,UKRAINIAN,RUSSIAN,Fight,"Kyiv, Kyyiv, Misto, Ukraine",Defence & Military | Conflict & Geopolitical Tensions | Energy Security,85.38,Critical
3,2026-08-24 00:00:00,RUSSIAN,UKRAINE,Fight,"Vede, Russia (general), Russia",Defence & Military | Conflict & Geopolitical Tensions,84.92,Critical
4,2026-08-24 00:00:00,MOLDOVAN,ROMANIA,Fight,"Vede, Russia (general), Russia",Defence & Military | Conflict & Geopolitical Tensions,84.37,Critical
5,2026-08-24 00:00:00,BULGARIAN,NaN,Fight,"Kremlin, Moskva, Russia",Cybersecurity | Defence & Military | Conflict & Geopolitical Tensions,83.46,Critical
6,2026-08-24 00:00:00,MOLDOVAN,UKRAINIAN,Fight,"Vede, Russia (general), Russia",Defence & Military | Conflict & Geopolitical Tensions,80.25,Critical
7,2026-08-24 00:00:00,BRITAIN,RUSSIA,Fight,"Kremlin, Moskva, Russia",Defence & Military | Energy Security,80.09,Critical
8,2026-08-24 00:00:00,MOLDOVAN,UKRAINIAN,Fight,"Vynohradivka, Odes'ka Oblast, Ukraine",Defence & Military | Conflict & Geopolitical Tensions,77.85,Critical
9,2026-08-24 00:00:00,RUSSIA,UKRAINE,Fight,"Odesa, Odes'ka Oblast, Ukraine",Defence & Military | Conflict & Geopolitical Tensions,77.85,Critical


In [165]:
query = """
SELECT
    event_root_label,
    COUNT(*) AS events,
    ROUND(
        AVG(attention_score),
        2
    ) AS avg_attention_score
FROM security_events
GROUP BY event_root_label
ORDER BY events DESC;
"""

event_types_sql = pd.read_sql_query(
    query,
    conn
)

event_types_sql.head(15)

,event_root_label,events,avg_attention_score
0,Consult,28,34.05
1,Fight,20,78.63
2,Make Public Statement,12,36.31
3,Express Intent to Cooperate,12,27.24
4,Provide Aid,8,29.62
5,Engage in Diplomatic Cooperation,8,34.26
6,Coerce,7,66.93
7,Appeal,5,38.85
8,Reject,4,53.97
9,Threaten,3,66.70


In [166]:
import plotly.express as px

print("Plotly ready")

Plotly ready


In [174]:
fig.show(renderer="browser")

In [175]:
attention_order = [
    "Low",
    "Medium",
    "High",
    "Critical"
]

attention_sql["attention_band"] = pd.Categorical(
    attention_sql["attention_band"],
    categories=attention_order,
    ordered=True
)

attention_sql = attention_sql.sort_values(
    "attention_band"
)

In [176]:
fig = px.bar(
    attention_sql,
    x="attention_band",
    y="events",
    title="Security Events by Attention Level",
    labels={
        "attention_band": "Attention Level",
        "events": "Number of Events"
    }
)

fig.show(renderer="browser")

In [177]:
print(attention_sql)

check_total = pd.read_sql_query(
    """
    SELECT COUNT(*) AS total_events
    FROM security_events;
    """,
    conn
)

print(check_total)

  attention_band  events
0            Low      50
1         Medium      30
2           High      20
3       Critical      14
   total_events
0           114


In [178]:
event_types_plot = event_types_sql.sort_values(
    "avg_attention_score",
    ascending=True
)

fig2 = px.bar(
    event_types_plot,
    x="avg_attention_score",
    y="event_root_label",
    orientation="h",
    title="Average Attention Score by Event Type",
    labels={
        "avg_attention_score": "Average Attention Score",
        "event_root_label": "Event Type"
    }
)

fig2.show(renderer="browser")

In [179]:
query = """
SELECT
    event_root_label,
    COUNT(*) AS events,
    ROUND(
        AVG(attention_score),
        2
    ) AS avg_attention_score
FROM security_events
GROUP BY event_root_label
ORDER BY events DESC;
"""

event_types_sql = pd.read_sql_query(
    query,
    conn
)

event_types_sql

,event_root_label,events,avg_attention_score
0,Consult,28,34.05
1,Fight,20,78.63
2,Make Public Statement,12,36.31
3,Express Intent to Cooperate,12,27.24
4,Provide Aid,8,29.62
5,Engage in Diplomatic Cooperation,8,34.26
6,Coerce,7,66.93
7,Appeal,5,38.85
8,Reject,4,53.97
9,Threaten,3,66.70


In [ ]:
fig3 = px.scatter(
    event_types_sql,
    x="events",
    y="avg_attention_score",
    size="events",
    hover_name="event_root_label",
    title="Event Frequency vs Average Attention Score",
    labels={
        "events": "Number of Events",
        "avg_attention_score": "Average Attention Score"
    }
)

fig3.show(renderer="browser")

In [181]:
fig3 = px.scatter(
    event_types_sql,
    x="events",
    y="avg_attention_score",
    size="events",
    hover_name="event_root_label",
    title="Event Frequency vs Average Attention Score",
    labels={
        "events": "Number of Events",
        "avg_attention_score": "Average Attention Score"
    }
)

fig3.show(renderer="browser")

## 29. Interactive Security Events Map

In [182]:
query = """
SELECT
    event_date,
    actor1,
    actor2,
    event_root_label,
    location,
    latitude,
    longitude,
    security_domains,
    attention_score,
    attention_band,
    source_url
FROM security_events
WHERE latitude IS NOT NULL
  AND longitude IS NOT NULL;
"""

map_data = pd.read_sql_query(
    query,
    conn
)

print("Events available for map:", len(map_data))

map_data.head()

Events available for map: 114


,event_date,actor1,actor2,event_root_label,location,latitude,longitude,security_domains,attention_score,attention_band,source_url
0,2026-08-17 00:00:00,GOVERNMENT SPOKESMAN,FRANCE,Make Public Statement,"Paris, France (general), France",48.8667,2.33333,Defence & Military,31.47,Low,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
1,2026-08-17 00:00:00,GOVERNMENT SPOKESMAN,FRENCH,Make Public Statement,"Paris, France (general), France",48.8667,2.33333,Defence & Military,31.47,Low,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
2,2026-08-17 00:00:00,GOVERNMENT SPOKESMAN,FRENCH,Make Public Statement,"Maroua, Extreme-Nord, Cameroon",10.5909,14.31590,Defence & Military,28.09,Low,https://www.egyptindependent.com/the-worlds-oldest-president-left-for-a-brief-stay-in-europe-he-hasnt-returned-in-two-months/
3,2026-08-17 00:00:00,POLAND,AMBASSADOR,Consult,"Warsaw, (PL67), Poland",52.2500,21.00000,Defence & Military,30.02,Low,https://forward.com/fast-forward/846473/israel-slammed-for-closing-probe-into-killing-of-seven-world-central-kitchen-workers/
4,2026-08-24 00:00:00,NaN,LONDON,Disapprove,"Kyiv, Kyyiv, Misto, Ukraine",50.4333,30.51670,Defence & Military,49.47,Medium,https://londonlovesbusiness.com/kremlin-warns-britain-over-burnhams-secret-storm-shadow-plans/


In [183]:
fig4 = px.scatter_map(
    map_data,
    lat="latitude",
    lon="longitude",
    size="attention_score",
    hover_name="location",
    hover_data={
        "actor1": True,
        "actor2": True,
        "event_root_label": True,
        "security_domains": True,
        "attention_score": True,
        "attention_band": True,
        "latitude": False,
        "longitude": False
    },
    zoom=3,
    center={
        "lat": 54,
        "lon": 15
    },
    title="European Security Events Monitor"
)

fig4.update_layout(
    map_style="open-street-map"
)

fig4.show(renderer="browser")

In [185]:
print("location_countries" in df_sql.columns)

True


In [186]:
df_sql["location_countries_text"] = (
    df_sql["location_countries"]
    .apply(
        lambda x: " | ".join(x)
        if isinstance(x, list)
        else ""
    )
)

In [187]:
df_sql[
    [
        "location",
        "location_countries",
        "location_countries_text"
    ]
].head(10)

,location,location_countries,location_countries_text
8,"Paris, France (general), France",[France],France
9,"Paris, France (general), France",[France],France
10,"Maroua, Extreme-Nord, Cameroon",[],
11,"Warsaw, (PL67), Poland",[Poland],Poland
18,"Kyiv, Kyyiv, Misto, Ukraine",[Ukraine],Ukraine
20,Turkey,[Türkiye],Türkiye
21,Ukraine,[Ukraine],Ukraine
22,"Odesa, Odes'ka Oblast, Ukraine",[Ukraine],Ukraine
30,"White House, District of Columbia, United States",[],
31,"White House, District of Columbia, United States",[],


In [188]:
df_sql_events = df_sql[
    security_event_columns
].copy()

print("Rows:", len(df_sql_events))
print("Columns:", len(df_sql_events.columns))

Rows: 114
Columns: 25


In [189]:
df_sql_events.to_sql(
    "security_events",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite table updated")

SQLite table updated


In [191]:
df_sql["location_countries_text"]

8       France
9       France
10            
11      Poland
18     Ukraine
        ...   
258    Ukraine
263    Ukraine
264    Ukraine
265    Ukraine
266           
Name: location_countries_text, Length: 114, dtype: str

In [192]:
security_event_columns = [
    "event_id",
    "event_date",

    "actor1",
    "actor1_country",
    "actor1_type",

    "actor2",
    "actor2_country",
    "actor2_type",

    "event_code",
    "event_root_code",
    "event_root_label",
    "quad_class",
    "quad_class_label",

    "goldstein_scale",
    "avg_tone",

    "num_mentions",
    "num_articles",

    "location",
    "location_countries_text",
    "latitude",
    "longitude",

    "security_domains",

    "attention_score",
    "attention_band",

    "source_name",
    "source_url"
]

In [193]:
df_sql_events = df_sql[
    security_event_columns
].copy()

In [194]:
df_sql_events.to_sql(
    "security_events",
    conn,
    if_exists="replace",
    index=False
)

print("SQLite table updated")
print("Rows:", len(df_sql_events))

SQLite table updated
Rows: 114


In [196]:
query = """
SELECT
    event_date,
    actor1,
    actor2,
    event_root_label,
    location,
    location_countries_text,
    latitude,
    longitude,
    security_domains,
    attention_score,
    attention_band,
    source_url
FROM security_events
WHERE latitude IS NOT NULL
  AND longitude IS NOT NULL
  AND location_countries_text IS NOT NULL
  AND location_countries_text <> '';
"""

map_data = pd.read_sql_query(
    query,
    conn
)

print("Events available for European map:", len(map_data))

Events available for European map: 86


In [197]:
fig4 = px.scatter_map(
    map_data,
    lat="latitude",
    lon="longitude",
    size="attention_score",
    hover_name="location",
    hover_data={
        "actor1": True,
        "actor2": True,
        "event_root_label": True,
        "security_domains": True,
        "attention_score": True,
        "attention_band": True,
        "latitude": False,
        "longitude": False
    },
    zoom=3,
    center={
        "lat": 54,
        "lon": 15
    },
    title="European Security Events Monitor"
)

fig4.update_layout(
    map_style="open-street-map"
)

fig4.show(renderer="browser")

In [198]:
%pip install streamlit

   ---------------------------------------- 0.0/10.5 MB ? eta -:--:--
   ---------- ----------------------------- 2.9/10.5 MB 18.7 MB/s eta 0:00:01
   -------------------------- ------------- 7.1/10.5 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------  10.5/10.5 MB 19.0 MB/s eta 0:00:01
   ---------------------------------------- 10.5/10.5 MB 17.4 MB/s  0:00:00
   ---------------------------------------- 0.0/797.6 kB ? eta -:--:--
   ---------------------------------------- 797.6/797.6 kB 13.1 MB/s  0:00:00
   ---------------------------------------- 0.0/28.6 MB ? eta -:--:--
   ---- ----------------------------------- 3.4/28.6 MB 17.0 MB/s eta 0:00:02
   ---------- ----------------------------- 7.3/28.6 MB 18.0 MB/s eta 0:00:02
   --------------- ------------------------ 11.0/28.6 MB 17.8 MB/s eta 0:00:01
   --------------------- ------------------ 15.2/28.6 MB 18.4 MB/s eta 0:00:01
   -------------------------- ------------- 19.1/28.6 MB 18.4 MB/s eta 0:00:01
   ----

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
